# 6: Naïve Bayes, Decision Trees, Random Forests and SVMs

**Author:** Dr Rob Lyon 

**Version:** 1.0

## Code & License
Copyright © 2026 Robert Lyon. All Rights Reserved.

This notebook and all associated materials are the intellectual property of the author.

Permission is granted solely to read, study, and analyse this material for personal educational purposes. No other rights are granted.

Without the prior written consent of the author, you may not:

* Copy, reproduce, redistribute, publish, transmit, or display this work in whole or in part.
* Modify, adapt, transform, translate, or create derivative works based on this material.
* Incorporate any portion of this work into another project, publication, product, model, dataset, or codebase.
* Use this material for commercial purposes.
* Remove or alter this copyright notice.

All intellectual property rights, including copyright and any derivative rights, remain exclusively vested in the author.

Access to this material does not constitute a transfer of ownership, license, or any other intellectual property rights except as expressly stated above.

---

**Note:**

This notebook is designed to be viewed from within JupyterLab. The Table of Contents and "Back to Table of Contents" links rely on JupyterLab's anchor handling to jump between sections.

If you open this notebook in another tool, such as PyCharm or Visual Studio Code, these anchor links may not work as expected. You can still navigate the notebook in those tools by using the headings directly, most editors provide an outline or contents panel that lists them for you.

---

## Table of Contents

1. [Introduction](#1.-Introduction)
2. [Useful Links](#2.-Useful-Links)
3. [Libraries and Environment Setup](#3.-Libraries-and-Environment-Setup)
4. [Two Philosophies - Generative vs Discriminative](#4.-Two-Philosophies---Generative-vs-Discriminative)
5. [Naïve Bayes](#5.-Naïve-Bayes)
    - [5.1 Bayes Theorem](#5.1-Bayes-Theorem)
    - [5.2 Implementation from Scratch](#5.2-Implementation-from-Scratch)
    - [5.3 Naïve Bayes with scikit-learn](#5.3-Naïve-Bayes-with-scikit-learn)
6. [Decision Trees](#6.-Decision-Trees)
    - [6.1 From Stumps to Trees](#6.1-From-Stumps-to-Trees)
    - [6.2 Decision Tree with scikit-learn](#6.2-Decision-Tree-with-scikit-learn)
7. [Random Forest](#7.-Random-Forest)
    - [7.1 From Greedy Trees to Random Ensembles](#7.1-From-Greedy-Trees-to-Random-Ensembles)
    - [7.2 Random Forests with scikit-learn](#7.2-Random-Forests-with-scikit-learn)
8. [. Support Vector Machines](#8.-Support-Vector-Machines)
    - [8.1 The Maximum Margin Hyperplane](#8.1-The-Maximum-Margin-Hyperplane)
    - [8.2 SVMs with scikit-learn](#8.2-SVMs-with-scikit-learn)
    - [8.3 Kernels - Non-Linear Boundaries in Higher Dimensions](#8.3-Kernels---Non-Linear-Boundaries-in-Higher-Dimensions)
9. [All Classifiers Side by Side](#9.-All-Classifiers-Side-by-Side)
10. [Summary](#10.-Summary)
11. [References](#11.-References)

---

## 1. Introduction

This notebook introduces some of the major families of classification algorithm. The aim is to build your practical understanding of four core classifiers: **Naïve Bayes**, **Decision Trees**, **Random Forests**, and **Support Vector Machines**. We'll implement some of these from first principles and discuss others using scikit-learn, and along the way we'll draw out the fundamental distinction between generative and discriminative models.

### Learning Objectives

By the end of this notebook you should be able to:

- Explain the difference between generative and discriminative classifiers and give an example of each.
- Derive the Naïve Bayes decision rule from Bayes' theorem and explain the naïve independence assumption.
- Implement a categorical Naïve Bayes classifier from scratch, including Laplace smoothing.
- Apply `GaussianNB` from scikit-learn and interpret the output of `classification_report`.
- Describe how a decision tree grows by recursively applying decision stumps and what information gain means.
- Train `DecisionTreeClassifier` with scikit-learn, visualise the tree, and explain how `max_depth` controls overfitting.
- Explain how Random Forests use bootstrap sampling and random feature selection to reduce overfitting.
- Interpret a feature importance plot from a Random Forest.
- Describe the SVM margin and explain the role of support vectors.
- Explain what a kernel function does and choose an appropriate kernel for a given dataset.
- Compare classifiers on the same dataset and articulate why model choice depends on data geometry.

---


---

## 2. Useful Links

| Link | Description |
|------|-------------|
| [scikit-learn: supervised learning](https://scikit-learn.org/stable/supervised_learning.html) | Top-level overview of all supervised learning algorithms available in scikit-learn, with links to each module's user guide. |
| [Naïve Bayes in scikit-learn](https://scikit-learn.org/stable/modules/naive_bayes.html) | Detailed documentation for all Naïve Bayes variants (`GaussianNB`, `MultinomialNB`, `BernoulliNB`), including the assumptions each variant makes about the feature distribution. |
| [Decision Trees in scikit-learn](https://scikit-learn.org/stable/modules/tree.html) | User guide for `DecisionTreeClassifier` and `DecisionTreeRegressor`, covering the CART algorithm, Gini impurity, entropy, and hyperparameter guidance. |
| [Ensemble methods in scikit-learn](https://scikit-learn.org/stable/modules/ensemble.html) | Documentation for Random Forests, Gradient Boosted Trees, and other ensemble methods, including a discussion of bagging and feature importance. |
| [SVMs in scikit-learn](https://scikit-learn.org/stable/modules/svm.html) | Comprehensive guide to `SVC`, kernel functions, the margin concept, and the regularisation parameter $C$, with practical tips on scaling and kernel choice. |
| [A Visual Introduction to Machine Learning](http://www.r2d3.us/visual-intro-to-machine-learning-part-1/) | An excellent interactive visualisation that walks through decision tree learning step by step, highly recommended if you find the tree-growing intuition hard to follow. |


---


---

## 3. Libraries and Environment Setup

🔙 [Back to Table of Contents](#Table-of-Contents)

### 3.1 Working Environment

To run the code in this notebook, you will need a Python environment with the required libraries listed below installed.

The easiest way to get started is to use the **Project IO** Docker container, which provides a fully configured and tested environment containing all required Python packages and supporting dependencies. Instructions for downloading and running the container can be found in the **project-io** GitHub repository:

https://github.com/scienceguyrob/project-io

Using the Docker container ensures that the notebooks run in exactly the same environment in which they were developed and tested.

If you prefer not to use Docker, a good alternative is to install the [Anaconda Distribution](https://www.anaconda.com/products/distribution). You can then create a new Python environment and install the required libraries using either `conda install` or `pip install`.

### 3.2 Libraries Used in This Notebook

All sections of this notebook require the libraries listed below.

| Library | Purpose | Documentation |
|---------|---------|---------------|
| **NumPy** (`numpy`) | Numerical arrays, random number generation, and mathematical operations. Used throughout for data creation and computation. | [numpy.org](https://numpy.org/doc/stable/) |
| **Matplotlib** (`matplotlib.pyplot`) | The standard Python plotting library. Used to produce all decision region visualisations and comparison plots. | [matplotlib.org](https://matplotlib.org/stable/) |
| **scikit-learn** (`sklearn`) | Provides `GaussianNB`, `DecisionTreeClassifier`, `RandomForestClassifier`, `SVC`, dataset generators, and model evaluation utilities. | [scikit-learn.org](https://scikit-learn.org/stable/) |
| **collections** (`Counter`) | Python built-in. Used in the from-scratch Naïve Bayes implementation to count feature value frequencies. | [docs.python.org](https://docs.python.org/3/library/collections.html#collections.Counter) |
| **ipywidgets** (`ipywidgets`) | Adds interactive UI controls (sliders, dropdowns, etc.) to Jupyter notebooks. We use it to create sliders for adjusting plot parameters in real time. | [ipywidgets.readthedocs.io](https://ipywidgets.readthedocs.io/en/stable/) |
| **ipympl** (`matplotlib widget` backend) | Enables interactive, live-updating Matplotlib figures inside Jupyter. Required for smooth slider updates without redrawing the entire plot. | [matplotlib.org/ipympl](https://matplotlib.org/ipympl/) |

### 3.3 Data

This notebook uses synthetic datasets saved to disk, which can be found in the `data` directory.

* **clouds.csv**: contains 1,800 synthetic cloud observations and can be found at `data/clouds.csv`. This is used in section 6.
* **clouds_complex.csv**: 6,000 synthetic cloud observations described by ten physical features, including non-linear relationships, correlated noise, and deliberate label noise, stored at `data/clouds_complex.csv`. Used in Section 7.3 to compare a decision tree against a random forest and must be generated before any cell that loads it.
* **credit.csv**: contains 5,000 credit prediction entries, can be found at `data/credit.csv`. 

### 3.4 Imports

All library imports for this notebook are placed in the single cell below. This is deliberate best practice: keeping all imports in one place makes it easy to see at a glance what is required and avoids confusing `NameError` exceptions that arise when an import cell is accidentally skipped.

> ⚠️ **You must run the cell below before executing any other code cell in this notebook.**

In [ ]:
# Listing 1.
# ─────────────────────────────────────────────────────────────────────────────
# All library imports for this notebook are placed here for clarity.
# Run this cell first before executing any code.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np                          # numerical arrays and random number generation
import pandas as pd                         # to load data
import matplotlib.pyplot as plt             # plotting and visualisation

import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50

from collections import Counter             # counting occurrences of discrete feature values

# scikit-learn: datasets
from sklearn.datasets import make_moons

# scikit-learn: model selection and evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# scikit-learn: classifiers used in this notebook
from sklearn.naive_bayes   import GaussianNB
from sklearn.tree          import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble      import RandomForestClassifier
from sklearn.svm           import SVC
from sklearn.linear_model  import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Shared random number generator for reproducibility
rng = np.random.default_rng(42)   # seed 42 ensures the same data every run

# Confirm successful import
print('Libraries loaded successfully.')


---


## 4. Two Philosophies - Generative vs Discriminative

🔙 [Back to Table of Contents](#Table-of-Contents)

All machine learning classifiers fall into one of two camps depending on *what they learn from the training data*. This distinction shapes not just how a model works internally, but what it can and cannot do, and how much data it needs to do it well.

### The Core Idea

Imagine you're trying to classify whether an email is spam or not. You have a dataset of emails, each labelled *spam* or *not spam*, and each described by a set of features (word counts, sender info, etc.). There are two fundamentally different strategies for building a classifier:

1. **Learn what spam *looks like* as a whole.** Build a full picture of how spam emails are distributed across all their features, and do the same for legitimate emails. Then, when a new email arrives, ask: "which type of email is this new one more likely to have come from?"

2. **Learn where the *boundary* is.** Don't worry about modelling either class in detail. Just find a line (or curve, or surface) in the feature space that reliably separates spam from not-spam, and use that to classify new emails.

The first strategy is **generative**. The second is **discriminative**.

---

### Generative Models

**Generative models** learn the **joint probability distribution** $P(\mathbf{X}, Y)$.

Breaking that notation down:
- $\mathbf{X}$ is a vector of features, e.g. $\mathbf{X} = [\text{word count}, \text{has attachment}, \text{sender known}]$
- $Y$ is the class label, e.g. *spam* or *not spam*
- $P(\mathbf{X}, Y)$ is the probability of seeing *that particular combination of features and label together* in the data

So the model asks: *"How likely is it that I would observe these exact features AND this class label at the same time?"* It builds a separate statistical picture (a distribution) of what the data looks like inside each class.

Because it learns the full distribution of data within each class, it can, at least in principle, **generate brand new synthetic examples** that look like real data. This is where the name comes from.

To actually *classify* a new data point $\mathbf{x}$, a generative model uses **Bayes' theorem** to flip the question around and compute $P(Y \mid \mathbf{X})$, the probability of each class label *given* the observed features:

$$P(Y \mid \mathbf{X}) = \frac{P(\mathbf{X} \mid Y)\, P(Y)}{P(\mathbf{X})}$$

Reading this in plain English:

> *"The probability that this data point belongs to class $Y$, given its features $\mathbf{X}$, depends on how likely those features are if we're already in class $Y$, scaled by how common class $Y$ is overall."*

- $P(\mathbf{X} \mid Y)$: the **likelihood**. How probable are these features, given this class?
- $P(Y)$: the **prior**. How common is this class in the training data overall?
- $P(\mathbf{X})$: the **evidence**. How common are these features overall? (This acts as a normalising constant so the probabilities sum to 1.)

The model picks whichever class $Y$ gives the highest resulting probability.

---

### Discriminative Models

**Discriminative models** skip modelling the data distribution entirely. Instead they learn $P(Y \mid \mathbf{X})$ **directly**, the probability of a label given the observed features, purely from the training examples.

Rather than asking *"what does each class look like?"*, they ask only: *"given this input, which class should I output?"*

In practice, this means learning a **decision boundary**: a line, curve, or higher-dimensional surface that divides the feature space into regions, one per class. Anything on one side gets labelled class A; anything on the other side gets labelled class B. The internal structure of either class is not the model's concern.

Think of it like a border on a map. The border tells you which country you're in, but it tells you nothing about the geography, culture, or population density of either country.

---

### Side-by-Side Comparison

| Property | Generative | Discriminative |
|----------|------------|----------------|
| What is learned | $P(\mathbf{X}, Y)$, the full joint distribution of features and labels | $P(Y \mid \mathbf{X})$, the probability of each label, given the features |
| How classification works | Apply Bayes' theorem to derive $P(Y \mid \mathbf{X})$ | Predict $P(Y \mid \mathbf{X})$ directly |
| Can generate new data? | Yes, it has a model of what each class looks like | No, it only knows about the boundary, not the classes themselves |
| Data requirements | Typically needs more data to estimate distributions reliably | Can work with less data |
| Handles high-dimensional data | Harder: more features means more combinations to estimate | Easier: only needs to find a boundary, not model everything |
| Examples | Naïve Bayes, Gaussian Mixture Models | Logistic Regression, Decision Trees, SVM |

---

### The Curse of Dimensionality: Why Generative Models Struggle

The key practical limitation of generative models is the **curse of dimensionality** (introduced in an earlier notebook).

To estimate $P(\mathbf{X}, Y)$ well, the model needs to have seen enough training examples covering *all the relevant combinations* of feature values and class labels. But the number of possible combinations grows **exponentially** with the number of features.

Consider a simple pizza-ordering example. Suppose you have just four binary features:

| Feature | Possible values |
|---|---|
| Ordered before? | Yes / No |
| Evening order? | Yes / No |
| Large size? | Yes / No |
| Paid by card? | Yes / No |

With four binary features and two class labels, there are $2^4 \times 2 = 32$ possible $(\mathbf{X}, Y)$ combinations. Add just a few more features and that number explodes. Most combinations will appear *zero times* in your training data, making their probabilities impossible to estimate reliably.

This is sometimes called the **data sparsity** problem: the data gets thinner and thinner as the number of dimensions grows, no matter how large your dataset is.

**Discriminative models sidestep this entirely.** They don't try to estimate the density of every region of the feature space; they only need to find a boundary that separates the classes. This is a much simpler task, and it scales far better to high-dimensional data.

---

### Which Should You Use?

Neither is universally better; they suit different situations:

- If you have **limited data** and a **low-dimensional** feature space, a generative model may work well and gives you useful probabilistic insight into each class.
- If you have **many features** or care most about **classification accuracy**, a discriminative model will usually perform better and be more practical.
- If you need to **generate synthetic data** (e.g. for data augmentation or anomaly detection), only a generative model can do this.


The two panels below apply both philosophies to exactly the same dataset: 120 points drawn from two Gaussian clusters.

**Left panel, generative model:** Rather than jumping straight to a classification decision, the model first builds a statistical picture of each class. The dashed ellipses represent $P(\mathbf{X} \mid Y=c)$, the distribution of feature values *given* a particular class. The inner ellipse captures roughly 68% of that class's data; the outer ellipse captures roughly 95%. Notice that the model has a rich internal description of each class: it knows where the data lives in feature space, how spread out it is, and what a "typical" example looks like. Classification is then a secondary step: apply Bayes' theorem to ask which class this new point was most likely generated from.

**Right panel, discriminative model (Logistic Regression):** The model ignores the internal structure of each class entirely. Instead it learns a single decision boundary, the dashed line, which is the set of points where $P(Y=A \mid \mathbf{X}) = P(Y=B \mid \mathbf{X}) = 0.5$. On either side of that line, one class is more probable than the other, shown by the faint shading. The model has no knowledge of the shape, spread, or density of either cluster, only where the boundary between them lies.

Notice that the boundary found by logistic regression sits neatly between the two clusters. This is the *optimal* boundary for this data, the one that minimises misclassification, and it was found purely by learning from the labels, without the model ever needing to understand what either class looks like internally.


In [ ]:
# Listing 2.
%matplotlib widget
from visualisations.Figure_1 import show
show()


---

## 5. Naïve Bayes

🔙 [Back to Table of Contents](#Table-of-Contents)

### 5.1 Bayes Theorem

Naïve Bayes is the textbook example of a generative classifier, a model that works by building a probabilistic picture of each class and then using probability theory to classify new points. Before we can understand the classifier, we need to understand the theorem it is built on: Bayes' theorem.

---

### A Quick Primer on Conditional Probability

Before introducing the theorem, it helps to be clear about what **conditional probability** means, because it appears everywhere in this section.

$P(A)$ means *"the probability of event A happening."*

$P(A \mid B)$ means *"the probability of A happening, given that we already know B has happened."* The vertical bar $\mid$ is read as *"given"*.

These two quantities can be very different. Knowing something about the world changes your probability estimate for the event. Conditional probability formalises this idea.

---

#### A Concrete Example

Suppose we have a small dataset of 20 emails, each labelled as spam or not spam, and each noting whether the word *"offer"* appears:

|  | Contains "offer" | Does not contain "offer" | Total |
|:--|:--:|:--:|:--:|
| **Spam** | 8 | 2 | 10 |
| **Not spam** | 1 | 9 | 10 |
| **Total** | 9 | 11 | 20 |

We can now ask two very different probability questions.

**Question 1: What is the probability that a randomly chosen email is spam?**

$$P(\text{spam}) = \frac{10}{20} = 0.5$$

This is an *unconditional* probability: we have not looked at the email at all, just the overall proportions in the dataset.

**Question 2: What is the probability that an email is spam, given that it contains the word "offer"?**

$$P(\text{spam} \mid \text{contains "offer"}) = \frac{8}{9} \approx 0.89$$

Here we restrict attention to only the 9 emails containing "offer" and ask what fraction of those are spam. The difference is striking: 0.5 versus 0.89. Observing a single word has nearly doubled our estimate of the probability of spam.

We can also ask the question in the other direction:

$$P(\text{contains "offer"} \mid \text{spam}) = \frac{8}{10} = 0.80$$

Notice that $P(\text{spam} \mid \text{contains "offer"})$ and $P(\text{contains "offer"} \mid \text{spam})$ are related but not the same: 0.89 versus 0.80. This distinction is at the heart of Bayes' theorem, which gives us a principled way to move between these two directions of conditioning.


---

### Bayes' Theorem

Now suppose you know $P(\text{contains "offer"} \mid \text{spam})$, how often the word "offer" appears in emails that are already known to be spam. But what you actually want is the reverse: $P(\text{spam} \mid \text{contains "offer"})$, given that you can see the word "offer" in a new email, how likely is it to be spam? These are related but different questions, and we have just seen from the table above that they give different answers (0.80 versus 0.89).

**Bayes' theorem** gives us a principled way to move between these two directions, to *reverse* the conditioning:

$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

Reading this left to right: *"the probability of A given B equals the probability of B given A, multiplied by how likely A was in the first place, divided by how likely B was overall."*

Applied to our email example:

$$P(\text{spam} \mid \text{contains "offer"}) = \frac{P(\text{contains "offer"} \mid \text{spam}) \cdot P(\text{spam})}{P(\text{contains "offer"})}$$

$$= \frac{0.80 \times 0.50}{0.45} \approx 0.89$$

This matches exactly what we calculated directly from the table, reassuring confirmation that the theorem is doing the right thing.

In the general classification context, we substitute $A = Y$ (a class label) and $B = \mathbf{X}$ (the observed features):

$$P(Y \mid \mathbf{X}) = \frac{P(\mathbf{X} \mid Y) \cdot P(Y)}{P(\mathbf{X})}$$

Each term has a name and a meaning:

| Term | Name | Plain English meaning |
|------|------|-----------------------|
| $P(Y \mid \mathbf{X})$ | **Posterior** | Probability of class $Y$ *after* observing features $\mathbf{X}$, this is what we want |
| $P(\mathbf{X} \mid Y)$ | **Likelihood** | How probable are these features, assuming we are already in class $Y$? |
| $P(Y)$ | **Prior** | How common is class $Y$ in the training data, before seeing any features? |
| $P(\mathbf{X})$ | **Marginal likelihood** | How common are these feature values overall? Acts as a scaling constant to ensure probabilities sum to 1 |

The intuition: the posterior is the likelihood scaled by the prior, then normalised. A class that is very common ($P(Y)$ is large) gets a head start; a class that explains the observed features well ($P(\mathbf{X} \mid Y)$ is large) gets a further boost. In our example, spam gets a head start of 0.5 (half the emails are spam) and then a boost from the fact that "offer" appears in 80% of spam emails. Together these push the posterior up to 0.89.

---

### The Decision Rule

Once we have computed the posterior for every possible class, the decision rule is straightforward: predict whichever class has the highest posterior probability:

$$\hat{y} = \underset{y}{\arg\max} \ P(Y = y \mid \mathbf{X})$$

The notation $\underset{y}{\arg\max}$ means *"the value of $y$ that makes the expression as large as possible"*. In plain English: *"go through every class, compute its posterior probability, and pick the winner."*

Expanding this using Bayes' theorem:

$$\hat{y} = \underset{y}{\arg\max} \ \frac{P(\mathbf{X} \mid Y = y) \cdot P(Y = y)}{P(\mathbf{X})}$$

Now notice that $P(\mathbf{X})$, the denominator, does not depend on $y$ at all. It is the overall probability of observing this particular set of features, and it is the same fixed number regardless of which class we are currently evaluating. When we are simply asking *"which class scores highest?"*, dividing every class score by the same constant cannot change the answer: the ranking is unaffected. So we can drop it entirely from the comparison:

$$\hat{y} = \underset{y}{\arg\max} \ P(\mathbf{X} \mid Y = y) \cdot P(Y = y)$$

Think of it like comparing three fractions that all share the same denominator: you only need to compare the numerators to find the largest one.

This does **not** mean $P(\mathbf{X})$ is unimportant in general. If you needed the actual posterior probability as a properly calibrated number, say, to report *"this email is 94.3% likely to be spam"*, you would need to divide by $P(\mathbf{X})$ to normalise the result. But for a classification decision, the likelihood multiplied by the prior is all we need.

---

The interactive explorer below lets you plug numbers directly into Bayes' theorem and watch the calculation update live. The three sliders map onto the quantities we have just computed from the email table above:

- **P(B | A), the likelihood:** how often does the evidence appear within a given class? In our example, this is $P(\text{contains "offer"} \mid \text{spam}) = 0.80$, the word "offer" appeared in 80% of spam emails.
- **P(A), the prior:** how common is this class overall, before looking at any features? In our example, $P(\text{spam}) = 0.50$, half of all emails in the dataset were spam.
- **P(B | ¬A), the false positive rate:** how often does the evidence appear in the *other* class? In our example, $P(\text{contains "offer"} \mid \text{not spam}) = 0.10$, "offer" appeared in only 1 of the 10 legitimate emails.

$P(B)$, the overall probability of observing the evidence regardless of class, is computed automatically from the other three values:

$$P(\text{contains "offer"}) = P(\text{"offer"} \mid \text{spam}) \cdot P(\text{spam}) + P(\text{"offer"} \mid \text{not spam}) \cdot P(\text{not spam})$$

$$= 0.80 \times 0.50 + 0.10 \times 0.50 = 0.45$$

The result shown, $P(A \mid B)$, is the posterior: given that the word "offer" appears, what is the probability this email is spam?

**Things to try:**

- Leave the sliders at the values from our worked example (P(B|A) = 0.80, P(A) = 0.50, P(B|¬A) = 0.10) and verify that the result matches the 0.89 we calculated from the table.
- Now lower P(A), the prior, to 0.05, as if spam were rare in this inbox. Watch the posterior drop sharply even though the likelihood is unchanged. A rare class needs much stronger evidence to overcome its low prior.
- Set P(B|¬A) very low (say 0.02): "offer" almost never appears in legitimate emails. Now the posterior jumps close to 1.0. A highly specific signal produces a very confident prediction.
- Set P(B|A) and P(B|¬A) to the same value, say both 0.5. The posterior collapses back to the prior. If the word is equally likely in both classes, it carries no information at all.

The right panel shows the full working: the symbolic formula, your current slider values substituted in, and the equivalent classification form with $A = Y$ and $B = \mathbf{X}$. This is exactly the calculation Naïve Bayes performs at prediction time, once for each class, picking the one with the highest posterior.


In [ ]:
# Listing 3.
%matplotlib widget
from visualisations.Figure_1a import show
show()

---

### How Naïve Bayes Actually Learns

This is where it helps to think concretely. Suppose you have a training dataset with $n$ features, say, a dataset of emails described by word counts, message length, number of links, and so on. Each row in the dataset is one email; each column is one feature; and each row has a label: *spam* or *not spam*.

**Training** a Naïve Bayes classifier means extracting two things directly from that dataset: no gradient descent, no iterative optimisation, just counting and summarising.

**1. The prior $P(Y)$ for each class**

Simply count how often each class appears in the training data:

$$P(Y = \text{spam}) = \frac{\text{number of spam emails}}{\text{total number of emails}}$$

If 200 of your 1,000 training emails are spam, then $P(Y = \text{spam}) = 0.2$. This is the model's baseline belief about each class before it has looked at a single feature.

**2. The likelihood $P(\mathbf{X} \mid Y)$ for each feature, within each class**

This is where the word **naïve** comes in. In reality, features are often correlated: long emails might be more likely to contain lots of links, for example. Modelling all those dependencies jointly would require an enormous amount of data (recall the curse of dimensionality).

Naïve Bayes sidesteps this by making a bold simplifying assumption: **all features are conditionally independent given the class label.** That is, once you know an email is spam, knowing the word count tells you nothing extra about the number of links. This assumption is almost never exactly true in practice, hence *naïve*, but it makes the maths tractable and, surprisingly, the classifier still works well in many real-world settings.

Under this assumption, the joint likelihood of all $n$ features factorises into a product of individual feature likelihoods:

$$P(\mathbf{X} \mid Y) = P(x_1 \mid Y) \cdot P(x_2 \mid Y) \cdot P(x_3 \mid Y) \cdots P(x_n \mid Y) = \prod_{i=1}^{n} P(x_i \mid Y)$$

The symbol $\prod$ means *"multiply together"*; it is the multiplicative equivalent of $\sum$ (sum). So instead of estimating one huge joint distribution over all features simultaneously, the model estimates $n$ small, independent distributions, one per feature, per class. In plain English: *the probability of observing the full set of features $\mathbf{X}$ given class $Y$ is equal to the probability of seeing the first feature given $Y$, multiplied by the probability of seeing the second feature given $Y$, multiplied by the third, and so on for all $n$ features*.



What this looks like in practice depends on the type of each feature:

- **Categorical features** (e.g. a word being present or absent, a country name, a product category): the model counts how often each possible value appears within each class and converts those counts into probabilities. For example, if the word *"offer"* appears in 120 out of 200 spam emails, then $P(x_{\text{"offer"}} = \text{present} \mid Y = \text{spam}) = 0.6$.

- **Numerical features** (e.g. email length, transaction amount, a sensor reading): recall that the goal is to estimate the likelihood $P(x_i \mid Y)$, how probable is this specific feature value, given a particular class? To answer that, we need to model the underlying distribution of that feature's values within each class. In reality, any feature could follow many possible distributions (uniform, skewed, bimodal, and so on), but for simplicity Naïve Bayes typically assumes a **Gaussian (normal) distribution** within each class (an assumption that can cause problems if the data is heavily skewed or multimodal, but works surprisingly well in practice). The model therefore looks at all training examples belonging to a given class, extracts the values of that feature, and finds the mean $\mu$ and standard deviation $\sigma$ that best describe their spread. These two numbers fully define a Gaussian curve for that feature within that class. Once we have that curve, we can evaluate the likelihood of any specific value by reading off the height of the curve at that point using the Gaussian PDF: a high curve means that value is very plausible for this class, a low curve means it is unusual. This is exactly what the interactive plot above demonstrates.

So for each feature, the model asks: *"looking only at training examples that belong to class $Y$, what does this feature's distribution look like?"* It stores that summary, either a table of counts or a mean and standard deviation, and that is the learned likelihood for that feature.

When a new data point arrives with $n$ features, the model looks up (or evaluates) the likelihood of each observed feature value under each class, then multiplies all $n$ likelihoods together along with the prior. Because of the independence assumption, this is valid to do one feature at a time: the model never has to consider combinations of features jointly, which is precisely what makes it so computationally efficient even when $n$ is large.

---

The histogram below shows the distribution of email lengths across 200 synthetic spam emails, with each bar representing how frequently emails of that length appeared in the training data. The blue curve is the Gaussian distribution fitted to that data, described entirely by two parameters: the mean $\mu$ (where the curve peaks) and the standard deviation $\sigma$ (how wide it is).

The height of the curve at any point is the **probability density**: it tells us how plausible a given email length is under this class. It is not a probability in the everyday sense (it can exceed 1), but it plays the same role: $x_1$ being high on the curve means that length is very characteristic of spam; being low on the curve means it is unusual for spam. This height is exactly $P(x_1 \mid Y = \text{spam})$, the likelihood we need for Naïve Bayes.

The orange marker shows the query value $x_1$, the email length we want to evaluate. The dashed horizontal line traces across from the curve to the y-axis, reading off the likelihood value at that point. The equation panel in the top right shows the full calculation with your current values substituted in.

**How to use the plot:**

- **Drag $\mu$** left and right to shift the curve. A good fit peaks near the centre of the histogram bars. Notice how a misaligned $\mu$ makes many observed email lengths look implausible.
- **Drag $\sigma$** to widen or narrow the curve. Too narrow and the model becomes overconfident, with only emails very close to the mean seeming plausible. Too wide and it becomes uninformative.
- **Click "Snap to best fit"** to set $\mu$ and $\sigma$ to the values actually computed from the data; these are the parameters the model learns during training.
- **Drag $x_1$** to evaluate the likelihood at different email lengths. Move it toward the peak and watch the likelihood rise; drag it into the tails and watch it collapse toward zero. This is the value that feeds directly into the Naïve Bayes calculation.

In [ ]:
# Listing 4.
%matplotlib widget
from visualisations.Figure_1b import show
show()

#### A Word of Caution: The Normality Assumption

Gaussian Naïve Bayes works well when the feature values within each class are roughly normally distributed, as we saw with email length, where most spam emails cluster around a typical length with natural variation either side. But this assumption is not always valid, and applying it blindly can lead the model to learn a distorted picture of the data.

Consider a second feature: the **hour of day** an email was sent. Spam emails are dispatched by automated bots running across data centres in every time zone, so there is no reason to expect them to cluster around any particular hour. The sending time is effectively **uniformly distributed**: every hour from midnight to 11pm is equally likely.

If we force a Gaussian onto this data, which is exactly what Gaussian Naïve Bayes does, the model will peak around midday and taper toward zero at night. This means it will systematically underestimate the likelihood of late-night or early-morning spam, and overestimate the likelihood of midday spam, simply because of an assumption that was never true in the first place.

The plot below makes this visible. The green dashed line is the **true likelihood**, flat across all hours, as it should be for uniform data. The red curve is what the **Gaussian model actually learns**. The gold shading at $x_1$ shows the error between the two.

You do not need to worry about this deeply right now. The key takeaway is simply this: **always check whether your features are roughly normally distributed within each class before using Gaussian Naïve Bayes.** If they are not, the model will still run and produce predictions, but those predictions will be based on a flawed likelihood estimate. In practice, this is often acceptable, since Naïve Bayes is surprisingly robust, but it is important to know the assumption is there.


In [ ]:
# Listing 5.
%matplotlib widget
from visualisations.Figure_1c import show
show()

**Back to prediction.**

**At prediction time**, for a new data point $\mathbf{x}$, the model computes:

$$\hat{y} = \underset{y}{\arg\max} \ P(Y = y) \cdot \prod_{i=1}^{n} P(x_i \mid Y = y)$$

In plain English: *"for each class, multiply together the prior and all the individual feature likelihoods, then pick the class with the largest result."*

This is fast, interpretable, and requires very little data to train, one of the reasons Naïve Bayes remains a strong baseline classifier despite its age and its simplifying assumption.

### There Is No Loss Function Here

You might be wondering: where is the loss function or the optimisation problem? The answer is that Naïve Bayes does not have one or solve one. It is a fundamentally different kind of learning. Models like linear and logistic regression learned by defining a measure of error (a loss function), and then iteratively adjusting their parameters to minimise it. Learning was a process of gradual improvement driven by an error signal.

Naïve Bayes works nothing like this. It is a **closed-form, count-based estimator**, meaning the parameters it needs (the priors and the likelihoods) can be computed directly and exactly from the training data in a single pass. There is no iteration, no gradient, and no loss surface. Training is essentially: *look at the data, count things, divide.*

In practical terms this has a significant upside: Naïve Bayes is extremely fast to train, even on large datasets, and requires very little memory. The tradeoff is that the model is constrained by its assumptions, particularly the naïve independence assumption, in a way that a loss-minimising model is not, because a loss-minimising model can adapt its parameters to compensate for whatever the data throws at it.


### Example: Classifying a Spam Email

To make the mechanics concrete, suppose we have trained a Naïve Bayes classifier on a small dataset of three emails. In practice you would use thousands, but three is enough to follow every calculation by hand.

---

#### The Training Data

| Email | Length (words) | Sender type | Label |
|-------|---------------|-------------|-------|
| 1 | 41 | Unknown | Spam |
| 2 | 38 | Unknown | Spam |
| 3 | 312 | Known contact | Not spam |

We have two classes: $Y = \text{spam}$ and $Y = \text{not spam}$.
We have two features: $x_1 = \text{email length (words)}$ and $x_2 = \text{sender type}$.

---

#### Step 1: Compute the Priors $P(Y)$

The prior is simply the proportion of each class in the training data:

$$P(Y = \text{spam}) = \frac{2}{3} \approx 0.667$$

$$P(Y = \text{not spam}) = \frac{1}{3} \approx 0.333$$

This tells the model that, before looking at any features, a randomly chosen email is twice as likely to be spam as not spam, purely because that is what the training data reflects.

---

#### Step 2: Learn the Likelihoods

This is the step where the model learns what each class *looks like* across each feature separately.

**Feature 1: Email length (numerical)**

For numerical features, Naïve Bayes fits a Gaussian distribution to the values within each class. Remember, a Gaussian is fully described by two numbers: the mean $\mu$ (the average value) and the standard deviation $\sigma$ (how spread out the values are around that average).

From our training data:

| Class | Emails | Mean length $\mu$ | Std dev $\sigma$ |
|-------|--------|-------------------|------------------|
| Spam | 1, 2 | $\mu = 40$ | $\sigma = 8$ |
| Not spam | 3 | $\mu = 310$ | $\sigma = 40$ |

> *Note: with only one not-spam training example we cannot compute a real standard deviation, so we use a plausible value here. In practice, with thousands of training examples, these estimates become stable and reliable.*

To evaluate the likelihood of a specific email length $x_1$ occurring given a class, we plug into the **Gaussian probability density function**:

$$P(x_1 \mid Y) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(x_1 - \mu)^2}{2\sigma^2}\right)$$

This looks intimidating but all it does is return a number that is large when $x_1$ is close to the mean $\mu$, and small when $x_1$ is far away. In other words: *"how well does this email length fit the typical length we saw for this class?"*


The interactive plot below lets you build an intuition for this formula before we apply it to real numbers.

The curve is the Gaussian PDF of email length for the **spam class**. Its peak sits at $\mu$, and its width is controlled by $\sigma$. The height of the curve at any point is the likelihood: a high curve means that value of $x_1$ is very plausible for this class; a low curve means it is unusual.

**Try the following:**

- **Drag $x_1$** (the orange marker) left and right. Watch the equation panel update with the fully computed result. Notice that the likelihood is highest when $x_1$ sits directly under the peak, and drops away as you move it further from $\mu$ in either direction.
- **Drag $\mu$** to shift the entire curve left or right. This is what changes when two classes have different average email lengths. A spam class centred at $\mu = 40$ and a legitimate class centred at $\mu = 310$ will return very different likelihoods for the same $x_1$.
- **Drag $\sigma$** to make the curve wider or narrower. A large $\sigma$ means the class is very spread out (many different email lengths are plausible), so the curve is flat and the peak is low. A small $\sigma$ means the class is tightly concentrated, so the peak is tall and the likelihood drops off sharply. Notice that the total area under the curve always stays equal to 1 regardless of $\sigma$; this is what the normalising coefficient $\frac{1}{\sqrt{2\pi\sigma^2}}$ ensures.

For a new email of length $x_1 = 42$ words, the spam class (with $\mu = 40$, $\sigma = 8$) will return a high likelihood because 42 is very close to the spam mean. The not-spam class (with $\mu = 310$, $\sigma = 40$) will return a likelihood close to zero because 42 words is extremely far from anything seen in legitimate emails during training. Try setting these values in the plot to see this for yourself.


In [ ]:
# Listing 6.
%matplotlib widget
from visualisations.Figure_2 import show
show()

The plot below is essentially the same as the plot above, except it shows the likelihood for the **not-spam class** ($\mu = 310$, $\sigma = 40$). Notice that with $x_1 = 42$ words, our query email, the orange marker sits far out in the left tail of the distribution, producing a likelihood value close to zero. The not-spam class has never seen emails this short during training, so a 42-word email is extremely implausible if it's **not spam**. Not impossible, just very, very implausible.

In [ ]:
# Listing 7.
%matplotlib widget
from visualisations.Figure_2a import show
show()

**Feature 2: Sender type (categorical)**

For categorical features, the model counts how often each category appeared within each class and divides by the total number of examples in that class:

*Within spam emails:*

| Sender type | Count | Likelihood |
|-------------|-------|------------|
| Unknown | 2 | $P(x_2 = \text{unknown} \mid Y = \text{spam}) = \frac{2}{2} = 1.0$ |
| Known contact | 0 | $P(x_2 = \text{known} \mid Y = \text{spam}) = \frac{0}{2} = 0.0$ |

*Within not-spam emails:*

| Sender type | Count | Likelihood |
|-------------|-------|------------|
| Unknown | 0 | $P(x_2 = \text{unknown} \mid Y = \text{not spam}) = \frac{0}{1} = 0.0$ |
| Known contact | 1 | $P(x_2 = \text{known} \mid Y = \text{not spam}) = \frac{1}{1} = 1.0$ |

> *Note: a zero likelihood is a problem, as it would wipe out the entire product regardless of the other features. In practice this is handled with **Laplace smoothing**, which adds a small count to every category so that nothing ever has zero probability. We keep zeros here to focus on the core mechanics.*

---

#### Step 3: Classify a New Email

A new email arrives: **42 words long**, from an **unknown sender**. Which class does the model predict?

We compute the unnormalised posterior for each class. Recall that $P(\mathbf{X})$ cancels out so we only need:

$$\text{score}(y) = P(Y = y) \cdot P(x_1 \mid Y = y) \cdot P(x_2 \mid Y = y)$$

**Score for spam:**

First, evaluate the Gaussian likelihood for $x_1 = 42$ words under the spam distribution ($\mu = 40$, $\sigma = 8$):

$$P(x_1 = 42 \mid Y = \text{spam}) = \frac{1}{\sqrt{2\pi \cdot 64}} \exp\!\left(-\frac{(42-40)^2}{128}\right) \approx 0.0488$$

Then multiply everything together:

$$\text{score}(\text{spam}) = P(Y=\text{spam}) \cdot P(x_1=42 \mid \text{spam}) \cdot P(x_2=\text{unknown} \mid \text{spam})$$
$$= 0.667 \times 0.0488 \times 1.0 \approx 0.0326$$

**Score for not spam:**

First, evaluate the Gaussian likelihood for $x_1 = 42$ words under the not-spam distribution ($\mu = 310$, $\sigma = 40$):

$$P(x_1 = 42 \mid Y = \text{not spam}) = \frac{1}{\sqrt{2\pi \cdot 1600}} \exp\!\left(-\frac{(42-310)^2}{3200}\right) \approx 0.0$$

Then multiply everything together:

$$\text{score}(\text{not spam}) = 0.333 \times 0.0 \times 0.0 \approx 0.0$$

**Decision:**

$$\hat{y} = \underset{y}{\arg\max} \ \text{score}(y) = \text{spam}$$

The spam score is far larger, so the model predicts **spam**, correctly. The email is short (close to the spam mean of 40 words) and comes from an unknown sender (a category that appeared only in spam during training). Both features point firmly in the same direction.

---

### Another Worked Example: Disease Screening

Bayes' theorem produces results that are genuinely counterintuitive until you have seen them worked through carefully. The classic illustration is a medical screening test.

Suppose a disease affects 0.3% of the population. A test for this disease is quite good: it correctly identifies 89% of people who are ill (*sensitivity*), and it correctly rules out 93% of people who are healthy (*specificity*, meaning it returns a false positive 7% of the time).

You take the test. It comes back positive. How worried should you be?

Most people's instinct is *"very worried, the test is 89% accurate."* But this intuition is wrong, and Bayes' theorem shows us exactly why.

The key insight is that **the prior probability matters enormously**. The disease is very rare: only 3 in every 1,000 people have it. So among the people who test positive, the vast majority are healthy people who triggered a false alarm, simply because there are so many more healthy people in the population to begin with.

The code cell below works through this calculation step by step. Pay close attention to the output: the result is likely to surprise you, and understanding *why* it comes out that way is exactly the kind of probabilistic reasoning that Naïve Bayes relies on.

In [ ]:
# Listing 8.
# Illustrate Bayes' Theorem with the classic medical screening example.
# A disease has 0.3% prevalence; the test has 89% sensitivity and 93% specificity.

p_ill              = 0.003   # P(ill)       — disease prevalence in the population
p_healthy          = 1 - p_ill
p_pos_given_ill    = 0.89    # P(+ | ill)   — sensitivity (true positive rate)
p_pos_given_healthy= 0.07    # P(+ | healthy) — false positive rate (1 - specificity)

# ── Joint probabilities ──────────────────────────────────────────────────
p_ill_and_pos     = p_ill     * p_pos_given_ill
p_ill_and_neg     = p_ill     * (1 - p_pos_given_ill)
p_healthy_and_pos = p_healthy * p_pos_given_healthy
p_healthy_and_neg = p_healthy * (1 - p_pos_given_healthy)

# ── Marginal probability of a positive test (law of total probability) ───
# P(+) = P(+ | ill)*P(ill)  +  P(+ | healthy)*P(healthy)
p_pos = p_ill_and_pos + p_healthy_and_pos

# ── Bayes' theorem: P(ill | +) = P(+ | ill) * P(ill) / P(+) ────────────
p_ill_given_pos = (p_pos_given_ill * p_ill) / p_pos

print('Bayes Theorem applied to the disease screening example')
print('=' * 55)
print(f'  P(ill)              = {p_ill:.3f}  (disease prevalence)')
print(f'  P(+ | ill)          = {p_pos_given_ill:.3f}  (sensitivity)')
print(f'  P(+ | healthy)      = {p_pos_given_healthy:.3f}  (false positive rate)')
print()
print('Joint probabilities:')
print(f'  P(ill     and +)    = {p_ill_and_pos:.5f}')
print(f'  P(ill     and -)    = {p_ill_and_neg:.5f}')
print(f'  P(healthy and +)    = {p_healthy_and_pos:.5f}')
print(f'  P(healthy and -)    = {p_healthy_and_neg:.5f}')
print()
print(f'  P(+)                = {p_pos:.5f}  (marginal probability of a positive test)')
print()
print('Applying Bayes\' theorem:')
print(f'  P(ill | +) = P(+ | ill) x P(ill) / P(+)')
print(f'             = {p_pos_given_ill} x {p_ill} / {p_pos:.5f}')
print(f'             = {p_ill_given_pos:.4f}  ({p_ill_given_pos:.1%})')
print()
print('Interpretation: even with a positive test, a patient is only')
print(f'{p_ill_given_pos:.1%} likely to actually have the disease.')
print('Why? Because the disease is so rare that false positives dominate.')

---

### 5.2 Implementation from Scratch

(Skip to section 5.3 if you don't want to see how it works internally.)

The two functions below implement a complete Naïve Bayes classifier for **categorical features** (for simplicity) using only Python built-ins and NumPy. The dataset is a toy fruit classification problem: given the colour and size of a piece of fruit, predict whether it is an apple, banana, or orange.

**Laplace smoothing** is applied during fitting. Without it, any feature value that never appears in the training data for a particular class would give a likelihood of zero, causing the entire posterior for that class to collapse to zero, even if all other features are strongly supportive. Adding a count of 1 to every feature-value-class combination prevents this.

In [ ]:
# Listing 9.
# ── Naïve Bayes classifier built from scratch ───────────────────────────
# Dataset: fruit classification from colour and size.
# Features:
#   X[:, 0] — colour  (0=green, 1=yellow, 2=orange)
#   X[:, 1] — size    (0=small, 1=large)
# Labels:   0=apple, 1=banana, 2=orange

X_fruit = np.array([
    [0, 0], [0, 0], [0, 1], [1, 0],   # apples  (green/small or green/large)
    [1, 1], [1, 1], [1, 0], [1, 1],   # bananas (yellow/large)
    [2, 1], [2, 1], [2, 0], [2, 1],   # oranges (orange/large)
])
y_fruit      = np.array([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2])
class_names  = {0: 'Apple', 1: 'Banana', 2: 'Orange'}
feature_names= ['Colour (0=green,1=yellow,2=orange)', 'Size (0=small,1=large)']


def fit_naive_bayes(X, y):
    """Fit a Naïve Bayes model on categorical data.

    Parameters
    ----------
    X : ndarray, shape (n_samples, n_features)
        Integer-coded categorical feature matrix.
    y : ndarray, shape (n_samples,)
        Integer class labels.

    Returns
    -------
    priors      : dict  {class -> P(Y=class)}
    likelihoods : dict  {class -> {feature_index -> {value -> P(X_j=v | Y=class)}}}
    classes     : ndarray of unique class labels
    """
    classes = np.unique(y)   # array of distinct class labels
    n       = len(y)         # total number of training samples

    # ── Step 1: class prior probabilities  P(Y = c) ──────────────────────
    priors = {c: np.sum(y == c) / n for c in classes}

    # ── Step 2: feature likelihoods  P(X_j = v | Y = c) ─────────────────
    # likelihoods[c][j][v] = estimated probability of feature j taking value v
    # given the sample belongs to class c.
    likelihoods = {}
    for c in classes:
        X_c = X[y == c]          # subset of rows belonging to class c
        likelihoods[c] = {}
        for j in range(X.shape[1]):
            col_vals  = X_c[:, j]              # all values of feature j in class c
            counts    = Counter(col_vals)       # count occurrences of each value
            total     = len(col_vals)           # number of samples in class c
            n_vals    = len(np.unique(X[:, j])) # number of distinct values (for smoothing)
            # Laplace smoothing: add 1 to every count; add n_vals to denominator
            # This prevents zero-probability issues for unseen feature values.
            likelihoods[c][j] = {
                v: (counts.get(v, 0) + 1) / (total + n_vals)
                for v in np.unique(X[:, j])
            }
    return priors, likelihoods, classes


def predict_naive_bayes(x_new, priors, likelihoods, classes):
    """Predict the class of a single new sample.

    Computes log P(Y=c) + sum_j log P(X_j | Y=c) for each class c
    and returns the class with the highest score.
    """
    log_posteriors = {}
    for c in classes:
        log_p = np.log(priors[c])          # start with the log prior
        for j, val in enumerate(x_new):
            # Retrieve the likelihood; fall back to a tiny value if unseen
            lhood  = likelihoods[c][j].get(val, 1e-9)
            log_p += np.log(lhood)          # accumulate log likelihood
        log_posteriors[c] = log_p
    # Return the class with the highest log-posterior score
    return max(log_posteriors, key=log_posteriors.get), log_posteriors


# Fit the model on the fruit dataset
priors, likelihoods, classes = fit_naive_bayes(X_fruit, y_fruit)

print('Fitted Naïve Bayes model')
print('=' * 45)
print('Class priors  P(Y):')
for c, p in priors.items():
    print(f'  P(Y={class_names[c]:<7}) = {p:.3f}')
print()
print('Feature likelihoods  P(X_j | Y=c)  [with Laplace smoothing]:')
for c in classes:
    print(f'  Class: {class_names[c]}')
    for j in range(2):
        print(f'    Feature {j} ({feature_names[j]}): {dict(likelihoods[c][j])}')

---

The cells above fit the model and print the learned priors and likelihoods. The cell below uses the fitted model to classify each training sample and report overall accuracy. It also classifies a new, unseen fruit, a yellow, large item, and shows the full log-posterior scores so we can see how confident the model is.

In [ ]:
# Listing 10.
# ── Step 1: Evaluate accuracy on training data ───────────────────────────────
# Iterate over each fruit in the training set, predict its class, and compare
# the prediction to the true label. Track the number of correct predictions.
correct = 0
print('Predictions on training data:')
print(f'  {"True label":<12} {"Predicted":<12} {"Correct?"}')
print('-' * 40)
for xi, yi in zip(X_fruit, y_fruit):
    pred, log_posts = predict_naive_bayes(xi, priors, likelihoods, classes)
    ok       = (pred == yi)
    correct += int(ok)
    tick     = 'v' if ok else 'x'   # 'v' for correct, 'x' for incorrect
    print(f'  {class_names[yi]:<12} {class_names[pred]:<12} {tick}')

# Print the training accuracy as a fraction and percentage.
print(f'\nTraining accuracy: {correct}/{len(y_fruit)} = {correct/len(y_fruit):.1%}')

# ── Step 2: Classify a new unseen fruit ──────────────────────────────────────
# Define a new fruit with features: yellow (1) and large (1).
x_new = np.array([1, 1])
pred_new, log_posts_new = predict_naive_bayes(x_new, priors, likelihoods, classes)

print()
print('New fruit: yellow (1), large (1)')
print('Log-posterior scores (higher = more probable):')

# Print the log-posterior scores for each class.
for c, lp in log_posts_new.items():
    print(f'  {class_names[c]:<10}: {lp:.4f}')

# Print the predicted class for the new fruit.
print(f'Predicted class: {class_names[pred_new]}')

---

### 5.3 Naïve Bayes with scikit-learn

For **continuous** features, scikit-learn's `GaussianNB` assumes that the feature likelihoods $P(X_j \mid Y=c)$ follow a Normal distribution for each class, fitting a mean $\mu_{jc}$ and variance $\sigma^2_{jc}$ from the training data rather than counting discrete values. This is known as **Gaussian Naïve Bayes**.

---

#### Training the model

Training a Naïve Bayes classifier in scikit-learn follows the same three-step pattern used in other notebooks:

```python
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

# 1. Split the data into training and test sets
#    test_size=0.2 reserves 20% of the data for evaluation
#    random_state fixes the random seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Create the model
model = GaussianNB()

# 3. Fit the model (this is where the learning happens)
#    For Naïve Bayes this means computing the class priors and
#    the per-feature, per-class means and variances from the training data
model.fit(X_train, y_train)
```

Unlike the gradient-descent models from Notebook 5, `fit()` here is a single pass through the data: no iterations, no loss surface. It simply computes and stores the statistics needed for Bayes' theorem.

---

#### What the model has learned

After calling `fit()`, the trained model exposes its learned parameters as attributes you can inspect directly:

```python
model.classes_        # the class labels the model was trained on
model.class_prior_    # P(Y=c) for each class (the prior probabilities)
model.theta_          # per-class, per-feature means: mu_{jc}
model.var_            # per-class, per-feature variances: sigma^2_{jc}
```

- `class_prior_` contains $P(Y = c)$ for each class $c$, estimated by counting class frequencies in the training data, exactly as described in Section 5.2.
- `theta_` is a matrix of shape `(n_classes, n_features)`. Each entry is the mean $\mu_{jc}$ of feature $j$ within class $c$, used as the centre of the Gaussian likelihood.
- `var_` is the same shape. Each entry is the variance $\sigma^2_{jc}$ of feature $j$ within class $c$, controlling the width of the Gaussian likelihood.

These are the exact quantities that get substituted into the Gaussian PDF formula from Section 5.2 when classifying a new point.

---

#### Making predictions and getting probabilities

```python
# Predict the class label for each test example
#   Returns the class with the highest posterior P(Y | X) for each row
y_pred = model.predict(X_test)

# Predict the posterior probability of each class for each test example
#   Returns a matrix of shape (n_samples, n_classes)
#   Each row sums to 1 (these are properly normalised posteriors)
y_proba = model.predict_proba(X_test)
```

`predict()` applies the decision rule $\hat{y} = \underset{y}{\arg\max}\ P(Y=y) \cdot \prod_{i=1}^{n} P(x_i \mid Y=y)$ internally and returns the winning class label.

`predict_proba()` goes one step further and divides by $P(\mathbf{X})$ to return fully normalised posterior probabilities, so you can report not just *which* class was predicted, but *how confident* the model is. For example, a row of `[0.03, 0.97]` means the model assigns 97% posterior probability to the second class.

---

#### Evaluating the model

```python
from sklearn.metrics import accuracy_score, classification_report

# Overall fraction of test examples classified correctly
accuracy = accuracy_score(y_test, y_pred)

# Detailed per-class breakdown of precision, recall, and F1 score
#   (these metrics are covered in depth in Notebooks 10 and 11)
print(classification_report(y_test, y_pred))
```

---

**Key hyperparameter:** `var_smoothing` adds a small fraction of the largest observed feature variance to all per-class variances. This acts as a regulariser and prevents numerical issues when a feature happens to have zero variance within a class, which would make the Gaussian PDF undefined (division by zero inside the $\sqrt{2\pi\sigma^2}$ term). The default value of `var_smoothing` is `1e-9`, but you can adjust it when creating the model:

```python
# A larger var_smoothing value adds more regularisation,
# useful if your data has features with very low variance,
# or if the model is overfitting to the training data
model = GaussianNB(var_smoothing=1e-6)
model.fit(X_train, y_train)
```

A simple way to find a reasonable value is to try a few candidates manually and compare their accuracy on the test set:

```python
for vs in [1e-11, 1e-9, 1e-7, 1e-5, 1e-3]:
    m = GaussianNB(var_smoothing=vs)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f'var_smoothing={vs:.0e}  →  accuracy={acc:.4f}')
```

In practice, Gaussian Naïve Bayes is relatively insensitive to `var_smoothing` unless your dataset has features with near-zero variance within a class. If you find the model performing poorly, it is far more likely to be caused by the naïve independence assumption being badly violated than by this parameter.


**When to use Naïve Bayes:**

- Training and prediction are very fast, even on large datasets.
- Performs well on high-dimensional data such as text (word counts).
- A strong baseline when features are approximately conditionally independent.
- A good choice when training data is limited.

**When to avoid it:**

- Highly correlated features violate the independence assumption and can degrade performance.
- Complex non-linear relationships between features and class label.
- Zero-frequency problem: a feature value unseen during training for a class assigns that class probability zero. Laplace smoothing (as used above) mitigates this for categorical data.

The code cell below applies `GaussianNB` to a synthetic 300 sample three-class tumour classification dataset and visualises the resulting decision regions. The data is designed to mimic the kind of structure you would see in a real medical classification problem, where different diagnoses occupy different regions of the feature space.

#### Dataset structure

| Variable | Values | Description |
|----------|--------|-------------|
| Biomarker 1 | Continuous, positive | A quantitative measurement taken from the biopsy sample, loosely analogous to a measure of cell size |
| Biomarker 2 | Continuous, positive | A second independent measurement, loosely analogous to a measure of cell irregularity |
| Diagnosis | 0, 1, or 2 | The clinical outcome: 0 = Benign, 1 = Malignant-A, 2 = Malignant-B |


#### Class definitions

| Label | Class name | Cluster centre | Std dev | Samples |
|-------|------------|---------------|---------|---------|
| `0` | Benign | (2, 2) | 1.0 | 100 |
| `1` | Malignant-A | (5, 5) | 1.0 | 100 |
| `2` | Malignant-B | (2, 7) | 1.0 | 100 |

Each class is sampled from a 2-D Gaussian distribution centred at its cluster centre. The standard deviation of 1.0 is the same for all three classes, meaning each cluster has roughly the same spread. Benign and Malignant-B share the same Feature 1 value (both centred at $x_1 = 2$) but are well separated along Feature 2. This makes the boundary between those two classes primarily vertical in the feature space, which is a useful thing to look for in the decision region plot below.


In [ ]:
# Listing 11.
# ── Gaussian Naïve Bayes with scikit-learn — synthetic tumour biopsy data ────
#
# We simulate a dataset of 300 patients, each described by two continuous
# biomarker measurements taken from a biopsy sample:
#
#   Biomarker 1 — loosely analogous to cell size
#   Biomarker 2 — loosely analogous to cell irregularity
#
# Each patient has one of three diagnoses:
#   0 — Benign:      low cell size (~2), low irregularity (~2)
#   1 — Malignant-A: elevated cell size (~5), elevated irregularity (~5)
#   2 — Malignant-B: normal cell size (~2), very high irregularity (~7)
#
# The values are drawn from Gaussian distributions — reflecting the real-world
# assumption that biomarker readings within a diagnostic group are approximately
# normally distributed around a typical value, with natural biological variation.

rng_nb = np.random.default_rng(3)   # fixed seed for reproducibility
n_nb   = 300                         # 300 patients total, 100 per diagnosis

# np.vstack stacks arrays row-wise to build the full (300, 2) feature matrix.
# Each call to rng_nb.normal(centre, std, size) draws 100 patients from a
# 2-D Gaussian centred at the typical biomarker values for that diagnosis.
X_nb = np.vstack([
    rng_nb.normal([2, 2], 1.0, (n_nb // 3, 2)),  # Benign:      low size, low irregularity
    rng_nb.normal([5, 5], 1.0, (n_nb // 3, 2)),  # Malignant-A: high size, high irregularity
    rng_nb.normal([2, 7], 1.0, (n_nb // 3, 2)),  # Malignant-B: low size, very high irregularity
])

# Build the corresponding label vector — 100 zeros, 100 ones, 100 twos.
y_nb = np.array([0] * (n_nb // 3) + [1] * (n_nb // 3) + [2] * (n_nb // 3))

# ── Train / test split ───────────────────────────────────────────────────────
# We hold back 25% of patients (75) as a test set — the model never sees these
# during training, so they give an honest estimate of real-world performance.
# random_state=0 ensures the same split every time the cell is run.
X_tr_nb, X_te_nb, y_tr_nb, y_te_nb = train_test_split(
    X_nb, y_nb, test_size=0.25, random_state=0
)

# ── Fit the Gaussian Naïve Bayes model ──────────────────────────────────────
# GaussianNB() creates the model with default var_smoothing=1e-9.
# .fit() computes the class priors P(Y=c) and the per-class, per-feature
# Gaussian parameters (mean and variance) from the training data in one pass —
# no iterative optimisation, no gradient descent.
gnb = GaussianNB()
gnb.fit(X_tr_nb, y_tr_nb)

# ── Inspect what the model has learned ──────────────────────────────────────
# theta_ contains the per-class feature means (mu_{jc}):
#   rows = classes, columns = features
# var_ contains the per-class feature variances (sigma^2_{jc}):
#   same shape as theta_
print('Learned class priors P(Y=c):')
for cls, prior in zip(['Benign', 'Malignant-A', 'Malignant-B'], gnb.class_prior_):
    print(f'  {cls:<14} {prior:.3f}')

print('\nLearned feature means per class (theta_):')
for cls, means in zip(['Benign', 'Malignant-A', 'Malignant-B'], gnb.theta_):
    print(f'  {cls:<14} Biomarker 1 mean = {means[0]:.2f},  Biomarker 2 mean = {means[1]:.2f}')

# ── Predict on the held-out test set ────────────────────────────────────────
# .predict() applies the Naïve Bayes decision rule to each test patient and
# returns the diagnosis with the highest posterior probability.
y_pred_nb = gnb.predict(X_te_nb)

# ── Evaluate performance ─────────────────────────────────────────────────────
# accuracy_score computes the fraction of test patients correctly diagnosed.
# classification_report gives a per-diagnosis breakdown — precision, recall,
# and F1-score are covered in detail in Notebooks 10 and 11.
print('\nGaussian Naïve Bayes — tumour diagnosis results')
print('=' * 50)
print(f'Test accuracy: {accuracy_score(y_te_nb, y_pred_nb):.1%}')
print()
print(classification_report(y_te_nb, y_pred_nb,
      target_names=['Benign', 'Malignant-A', 'Malignant-B']))

# ── Decision region plot ─────────────────────────────────────────────────────
# To visualise what the model has learned, we create a fine grid of artificial
# patients covering the entire biomarker space, predict a diagnosis for every
# point on the grid, then colour each region by its predicted diagnosis.
# This reveals the decision boundaries the model has learned from the data.

h = 0.05   # grid step size in biomarker units — smaller = finer grid but slower

# Compute grid extent from the data, adding 1 unit of padding on each side
# so the boundaries are visible at the edges of the plot.
x_min, x_max = X_nb[:, 0].min() - 1, X_nb[:, 0].max() + 1
y_min, y_max = X_nb[:, 1].min() - 1, X_nb[:, 1].max() + 1

# np.meshgrid creates a dense grid of (Biomarker 1, Biomarker 2) coordinates.
# np.arange(start, stop, step) produces the coordinate values along each axis.
xx, yy = np.meshgrid(
    np.arange(x_min, x_max, h),
    np.arange(y_min, y_max, h)
)

# Predict the diagnosis at every grid point.
# xx.ravel() flattens the 2-D grid to a 1-D array so predict() can process it;
# np.c_[...] pairs up the x and y coordinates into (Biomarker1, Biomarker2) rows;
# .reshape(xx.shape) maps the predictions back onto the 2-D grid for plotting.
Z = gnb.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))

# contourf with colors= maps colours to contour levels, not class values,
# which can produce incorrect or missing regions for multi-class problems.
# cmap= maps the full colourmap continuously across the Z values (0, 1, 2),
# ensuring each class gets a distinct shaded region reliably.
ax.contourf(xx, yy, Z, alpha=0.18, cmap=plt.cm.Set1)
ax.contour(xx,  yy, Z, colors='white', linewidths=0.8, linestyles='--')

# Overlay the actual patient data points — training and test combined —
# so users can see how well the decision regions align with the true clusters.
colours = ['steelblue', 'tomato', 'seagreen']
labels  = ['Benign', 'Malignant-A', 'Malignant-B']
for cls, col, lbl in zip([0, 1, 2], colours, labels):
    mask = y_nb == cls   # boolean index selecting only patients with this diagnosis
    ax.scatter(X_nb[mask, 0], X_nb[mask, 1],
               color=col, s=35, edgecolors='k', lw=0.3,
               alpha=0.8, label=lbl, zorder=3)

ax.set_xlabel('Biomarker 1 (cell size)')
ax.set_ylabel('Biomarker 2 (cell irregularity)')
ax.set_title('Figure 3. Gaussian Naïve Bayes — decision regions\n'
             'Each shaded region is classified as one tumour diagnosis')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---

#### Interpreting the output

**Class priors**

The three priors are almost identical, approximately 0.33 each, which makes sense because we generated exactly 100 patients per class. The model has correctly learned that no diagnosis is inherently more common than the others in this dataset. In a real clinical setting these would differ: benign cases are typically far more common than malignant ones, and the prior would reflect that, pushing the model toward benign predictions unless the biomarkers strongly suggest otherwise.

**Learned feature means**

The model has recovered the cluster centres used to generate the data almost exactly:

| Diagnosis | True centre | Learned means |
|-----------|------------|---------------|
| Benign | (2.0, 2.0) | (2.02, 2.01) |
| Malignant-A | (5.0, 5.0) | (5.14, 5.06) |
| Malignant-B | (2.0, 7.0) | (1.95, 7.30) |

This is reassuring: it confirms that the model's Gaussian likelihood estimates are a faithful representation of the training data. Notice that Benign and Malignant-B have very similar Biomarker 1 means (~2.0) but very different Biomarker 2 means (2.0 vs 7.3). This means the model must rely almost entirely on Biomarker 2 to distinguish between these two diagnoses. A patient with low cell irregularity is classified as Benign, while one with high cell irregularity is classified as Malignant-B, even if their cell size readings are identical.

**Test accuracy**

The model correctly diagnosed 70 out of 75 held-out patients, achieving an overall accuracy of 93.3%. For a simple model trained in a single pass with no iterative optimisation, this is a strong result, and it is largely a consequence of the three clusters being well separated in the feature space.

**Per-class breakdown**

The classification report gives a more detailed picture than accuracy alone:

- **Precision**: of all patients the model *predicted* as having this diagnosis, what fraction actually had it? A precision of 0.92 for Benign means that 92% of patients labelled Benign by the model were genuinely Benign.
- **Recall**: of all patients who *truly* had this diagnosis, what fraction did the model correctly identify? A recall of 0.96 for Benign means the model caught 96% of all genuine Benign cases.
- **F1-score**: the harmonic mean of precision and recall, giving a single balanced summary of both. Values above 0.9 across all three classes indicate the model is performing consistently well.

Malignant-A has the lowest recall (0.88), meaning it is the hardest class to detect reliably. Some Malignant-A patients are being misclassified, most likely as Malignant-B or Benign. This is consistent with what we can see in the decision region plot: Malignant-A occupies a central position in the feature space, making its boundaries with the other two classes the most ambiguous.

> **A note on metrics:** precision, recall, and F1-score are covered in full detail in Notebook 11. For now, the key takeaway is that accuracy alone can be misleading. The per-class breakdown tells you *where* the model is making mistakes, which matters far more than the headline number in any real application.

#### Naïve Bayes Functions in scikit-learn

In scikit-learn, the Naïve Bayes family of classifiers is implemented in several variants, each optimised for a specific type of input data. The choice of variant depends on the nature of your features (continuous, discrete, binary, or categorical) and the assumptions you are willing to make about their underlying distribution.

Below is a summary of the available Naïve Bayes classifiers in scikit-learn, their supported data types, the probabilistic distribution they assume, their key hyperparameters, and additional notes for practical use.

| Variant | Data Type | Distribution | Key Hyperparameters | Notes | Best For |
|---------|-----------|--------------|----------------------|-------|----------|
| `GaussianNB` | Continuous | Normal (Gaussian) | `var_smoothing` (default: `1e-9`) | Estimates mean (`theta_`) and variance (`var_`) for each feature per class. | Continuous features where each class's feature values are approximately normally distributed (e.g., height, weight, biomarker measurements). |
| `MultinomialNB` | Discrete counts | Multinomial | `alpha` (default: `1.0`), `fit_prior` (default: `True`) | Requires non-negative feature values. Uses additive smoothing. | Discrete count data, such as word frequencies in text classification or other non-negative integer features. |
| `BernoulliNB` | Binary/Boolean | Bernoulli | `alpha` (default: `1.0`), `fit_prior` (default: `True`), `binarize` (default: `0.0`) | Converts continuous features to binary using `binarize` threshold. | Binary or boolean features (e.g., presence/absence of words, binary indicators). |
| `ComplementNB` | Discrete counts | Complement Multinomial | `alpha` (default: `1.0`), `norm` (default: `False`) | Uses complement probabilities for imbalanced datasets. | Imbalanced datasets with discrete count features, as it uses complement probabilities for stability. |
| `CategoricalNB` | Categorical | Categorical | `alpha` (default: `1.0`), `categories`, `min_categories` (default: `None`) | Requires `categories` parameter for unique feature categories. | Categorical features (e.g., labels, categories) where each feature represents a distinct category. |


---
## 6. Decision Trees

🔙 [Back to Table of Contents](#Table-of-Contents)

So far we have looked at Naïve Bayes, a generative classifier that works by building a probabilistic model of each class and applying Bayes' theorem to classify new points. We now turn our attention to **discriminative** approaches, which take a fundamentally different strategy: rather than modelling what each class looks like, they focus entirely on finding boundaries that separate the classes.

We start with perhaps the most intuitive discriminative classifier of all, the **decision tree**.

---

### 6.1 From Stumps to Trees

#### The Decision Stump: a Single Split

In a previous notebook we introduced the idea of finding a threshold that splits data into two groups. The problem was simple: classify fruit as either an apple or an orange based on diameter alone. The approach was to find a value $\theta$ (theta) such that:

$$\hat{y} = \begin{cases} \text{Orange} & \text{if diameter} \geq \theta \\ \text{Apple} & \text{if diameter} < \theta \end{cases}$$

The process for finding $\theta$ was essentially trial and error: try different threshold values, count the misclassifications each one produces, and pick the value that makes the fewest mistakes. Simple, interpretable, and surprisingly effective on clean, low-dimensional data.

What we described there was actually a **decision stump**, the simplest possible decision model. Formally:

> A **decision stump** is a model that applies a single threshold $\theta$ on a single feature to partition data into two classes.





In [ ]:
# Listing 12.
%matplotlib widget
from visualisations.Figure_4a import show
show()

The interactive figure below brings the decision stump to life on the fruit classification problem. Rather than finding the optimal threshold automatically, you are in control: drag the threshold $\theta$ through the data and watch what happens.

**Left panel, the data:** each point is one piece of fruit, positioned along the x-axis by its diameter. Blue points are apples; red points are oranges. The dashed vertical line is your current threshold: everything to its left is predicted Apple, everything to its right is predicted Orange. Any point on the wrong side of the line is highlighted in **gold**; these are the misclassifications your current threshold produces.

Notice that no matter where you place the threshold, some points will always be misclassified. This is because the two classes overlap in diameter: some apples are larger than some oranges, and no single vertical line can perfectly separate them. This is a fundamental limitation of the decision stump: it can only ask one question, and sometimes one question is not enough.

**Right panel, the error landscape:** rather than just showing the errors for your current threshold, this panel shows the misclassification count for *every possible* threshold value. The curve reveals the full picture: where errors are high, where they drop, and crucially, where the minimum lies. The green dashed line marks the optimal threshold, the value a decision stump learning algorithm would find automatically by searching this landscape.

In [ ]:
# Listing 13.
%matplotlib widget
from visualisations.Figure_4b import show
show()

The stump has one obvious limitation: it can only ask one question. If the answer to that question does not cleanly separate the classes, for example, if apples and oranges overlap in diameter, the stump is stuck. It cannot refine its answer.

#### Extending to a Full Decision Tree

A **decision tree** overcomes this by chaining stumps together into a hierarchy. After the first split divides the data into two groups, each group is handed to another stump, which asks a new question, possibly on a different feature, with a different threshold. That stump splits its group again, and so on, until the tree decides it has learned enough.

The result is a structure of **nodes** and **branches**:

- Each **internal node** is a decision stump: it asks a yes/no question about one feature, *"is feature $x_j \geq \theta$?"*
- Each **branch** represents the answer, yes (right) or no (left), and directs a data point to the next node
- Each **leaf node** is a final answer: the class label assigned to any data point that reaches it

A data point is classified by starting at the top of the tree, the **root node**, and following the branches that match its feature values, one node at a time, until it reaches a leaf. The label at that leaf is the prediction.

This hierarchical chain of binary questions is what makes decision trees so appealing: the reasoning is entirely transparent. You can trace exactly why any prediction was made, step by step, which is rare among machine learning models. A doctor, a judge, or a regulator can read a decision tree and understand it, something that cannot be said for most of the models we will encounter in later notebooks.


The diagram below shows the structure of a decision tree built on two features, fruit diameter and a colour score. Trace any path from the root node at the top down to a leaf at the bottom: each node asks one yes/no question, each branch follows the answer, and the leaf gives the final prediction.

In [ ]:
# Listing 14.
%matplotlib widget
from visualisations.Figure_5 import show
show()

---

A tree is a chain of stumps, linked together such that together they classify the data as accurately as possible. Each node asks a question about one feature, and the answer routes a data point either left or right. The tree keeps splitting until it is confident enough in its predictions, or until it runs out of useful questions to ask.

The challenge, of course, is deciding **which feature to split on at each node, and where to place the threshold**. With even a modest number of features, there are an enormous number of possible splits to consider. The tree needs a principled way to evaluate them: a score that says "this split, on this feature, is more useful than that one."

---

#### How Do We Measure a Good Split?

Intuitively, a good split is one that produces groups that are **purer** than the group we started with. If we split a bag of mixed apples and oranges into two bags, one containing mostly apples and one containing mostly oranges, we have made progress. If we split it into two bags that are both still mixed in roughly equal proportions, we have learned nothing.

The question is: how do we measure *purity* mathematically?

---

#### Entropy: Measuring Disorder

The standard answer in decision trees is **entropy**, a concept borrowed from information theory. Entropy measures the **amount of disorder or uncertainty** in a set of labels. A set where every example belongs to the same class has zero entropy, since there is no uncertainty. A set where the classes are perfectly evenly mixed has maximum entropy, since we have no idea which class to predict.

For a set of data points with $K$ classes, entropy is defined as:

$$H = - \sum_{k=1}^{K} p_k \log_2(p_k)$$

where $p_k$ is the proportion of examples belonging to class $k$. The $\log_2$ means we measure entropy in **bits**, the amount of information needed to describe the uncertainty.

Breaking this down:
- $p_k$: the fraction of points in this group that belong to class $k$
- $\log_2(p_k)$: a large negative number when $p_k$ is small (rare classes contribute more surprise)
- The minus sign out front makes the whole expression positive
- We sum this over all classes

**Two special cases are worth understanding intuitively:**

If all points belong to one class, say $p_\text{apple} = 1.0$ and $p_\text{orange} = 0.0$, then $H = 0$. The group is perfectly pure. There is no uncertainty about what to predict.

If the classes are split equally, $p_\text{apple} = 0.5$ and $p_\text{orange} = 0.5$, then $H = 1.0$ bit. The group is maximally uncertain. A coin flip would do just as well as any model.

The interactive plot below shows how entropy changes as the composition of a group changes, in this case, a group of fruit containing some mix of apples and oranges.

The x-axis is $p_\text{orange}$, the proportion of oranges in the group, ranging from 0 (all apples) to 1 (all oranges). The y-axis is the entropy $H$, measured in bits.

**Drag the slider** to change the class proportions and watch three things update live:

- The **red marker** moves along the curve showing the current entropy value
- The **equation panel** substitutes your current values into the formula and computes the result step by step
- The **title** updates with the exact entropy in bits

**Key things to notice:**

- At $p_\text{orange} = 0$ (all apples) and $p_\text{orange} = 1$ (all oranges) the entropy is exactly **0 bits**: the group is perfectly pure. There is no uncertainty about what to predict; every example belongs to the same class.
- At $p_\text{orange} = 0.5$ (equal mix) the entropy reaches its maximum of **1.0 bit**: the group is maximally uncertain. Knowing nothing else, you would do no better than a coin flip.
- The curve is **symmetric**: a group that is 30% orange has the same entropy as one that is 70% orange. What matters is the degree of mixing, not which class dominates.

The entropy of a node is what the decision tree is trying to **reduce** with each split. Before a split, the parent node has some entropy. After the split, each child group should ideally be purer (lower entropy) than the parent was. The greater that reduction, the more useful the split.

In [ ]:
# Listing 15.
%matplotlib widget
from visualisations.Figure_6 import show
show()

Any real group of training data will have entropy somewhere between these two extremes. A good split is one that **reduces entropy**: it takes a high-entropy parent group and divides it into lower-entropy children. The greater the reduction, the more useful the split.

This reduction is called **information gain**, and it is the quantity a decision tree algorithm maximises when searching for the best split at each node.


#### Information Gain

Information gain answers a simple question: **how much does a proposed split reduce the uncertainty in our labels?**

Before any split, the full training set at a node has some entropy $H(\text{parent})$. After splitting on a feature, the data is divided into two child groups, left and right. Each child has its own entropy, but crucially, the two children may be of different sizes. A split that sends 90% of the data one way and 10% the other should be evaluated differently from one that splits the data evenly.

To account for this, we compute the entropy of the children as a **weighted average**: each child's entropy is weighted by the proportion of data points that fell into it:

$$H(\text{children}) = \frac{n_\text{left}}{n} \cdot H(\text{left}) + \frac{n_\text{right}}{n} \cdot H(\text{right})$$

where $n$ is the total number of points in the parent, $n_\text{left}$ and $n_\text{right}$ are the number of points that went left and right respectively, and $H(\text{left})$ and $H(\text{right})$ are the entropies of each child group.

Information gain is then the difference between the parent entropy and this weighted child entropy:

$$\text{IG} = H(\text{parent}) - \left( \frac{n_\text{left}}{n} \cdot H(\text{left}) + \frac{n_\text{right}}{n} \cdot H(\text{right}) \right)$$

In plain English: *"how much less uncertain are we about the labels after making this split, compared to before?"*

- A large information gain means the split has done real work: the children are substantially purer than the parent, and we have learned something useful.
- A small information gain means the split has barely helped: the children are about as mixed as the parent was.
- A gain of zero means the split was completely useless: the children are no purer than the parent.

---

##### A Concrete Example

Suppose a node contains 10 fruit, 5 apples and 5 oranges. The parent entropy is:

$$H(\text{parent}) = -0.5 \log_2 0.5 - 0.5 \log_2 0.5 = 1.0 \text{ bit}$$

We consider splitting on diameter at $\theta = 7.5$ cm. This sends 6 fruit to the left (5 apples, 1 orange) and 4 fruit to the right (0 apples, 4 oranges).

Left child entropy:

$$H(\text{left}) = -\frac{5}{6} \log_2 \frac{5}{6} - \frac{1}{6} \log_2 \frac{1}{6} \approx 0.650 \text{ bits}$$

Right child entropy:

$$H(\text{right}) = -\frac{0}{4} \log_2 \frac{0}{4} - \frac{4}{4} \log_2 \frac{4}{4} = 0.0 \text{ bits}$$

The right child is perfectly pure, all oranges. The weighted child entropy is:

$$H(\text{children}) = \frac{6}{10} \times 0.650 + \frac{4}{10} \times 0.0 = 0.390 \text{ bits}$$

Information gain:

$$\text{IG} = 1.0 - 0.390 = \mathbf{0.610 \text{ bits}}$$

The split has reduced our uncertainty by 0.610 bits, a substantial improvement. The decision tree algorithm evaluates splits like this across every feature and every candidate threshold, and selects the one with the highest information gain. That becomes the next node in the tree.

The IG formula introduced earlier can be written more compactly using standard notation. Rather than spelling out left and right explicitly, we write the weighted sum over all child nodes $v$ produced by the split:

$$\text{IG}(D, f) = H(D) - \sum_{v} \frac{|D_v|}{|D|} \cdot H(D_v)$$

where:

| Symbol | Meaning |
|--------|---------|
| $D$ | the dataset at the current node, the parent group |
| $f$ | the feature and threshold being evaluated for the split |
| $H(D)$ | the entropy of the parent group before the split |
| $v$ | an index over the child nodes produced by the split (left and right) |
| $D_v$ | the subset of data that falls into child node $v$ |
| $|D_v|$ | the number of data points in child $v$ |
| $|D|$ | the total number of data points in the parent |
| $|D_v| / |D|$ | the weight, the proportion of data that went to child $v$ |
| $H(D_v)$ | the entropy of child $v$ after the split |

The interactive figure below applies the information gain calculation directly to the fruit dataset. Drag the threshold $\theta$ to any position and watch both panels update live.

**Left panel, the split:** the data strip shows how the current threshold divides the fruit into two child groups. The label on each side shows the class counts and the entropy of that child group. Notice how moving the threshold toward a purer split, more apples on one side, more oranges on the other, causes the child entropies to drop.

**Right panel, the calculation:** the five steps of the information gain formula are shown with your current values substituted in:

- **Step 1, Parent entropy:** the entropy of the full group before any split is made. With 40 apples and 40 oranges this starts at 1.0 bit, maximum uncertainty.
- **Step 2, Left child entropy:** the entropy of the points that fell to the left of $\theta$ (diameter $< \theta$). A group containing mostly apples will have low entropy; a mixed group will have high entropy.
- **Step 3, Right child entropy:** the same calculation for the points that fell to the right (diameter $\geq \theta$).
- **Step 4, Weighted child entropy:** the two child entropies are combined into a single number, weighted by how many points went to each side. A large child group contributes more to this average than a small one.
- **Step 5, Information gain:** the parent entropy minus the weighted child entropy. This is the reduction in uncertainty the split has achieved. The larger this number, the more useful the split.

Drag $\theta$ to the far left or far right: almost all points go to one side, the other child is tiny, and the IG drops. An unbalanced split is rarely useful. 

In [ ]:
# Listing 16.
%matplotlib widget
from visualisations.Figure_7 import show
show()

So far we have established that a good split is one that reduces disorder in the class labels, and that entropy gives us a way to quantify that disorder. Information gain then tells us how much a particular split reduces the disorder. Together, entropy and information gain give the decision tree a solid basis for choosing which question to ask at each node (what do I split on?).

For **categorical features**, where a feature can only take a fixed set of values, such as sender type being "known", "unknown", or "mailing list", evaluating every possible split is straightforward. There are only a finite number of distinct values, so the algorithm simply tries each one and picks the split with the highest information gain.

For **numerical features**, such as email length, diameter, or a sensor reading, the problem is harder. A continuous feature can in principle take infinitely many values, so we cannot try them all. Instead, decision tree algorithms use the following practical approach:

1. **Sort** the training examples by the feature value in ascending order
2. **Generate candidate thresholds** that might be useful for splitting, by taking the midpoint between every pair of consecutive distinct values: if the sorted values are 6.1, 6.4, 7.2, the candidates are 6.25 and 6.8
3. **Evaluate each candidate** by placing the threshold at that midpoint and counting how many training samples fall on each side. For each resulting child group, the algorithm counts the class labels present and computes the proportion belonging to each class, then uses those proportions to calculate the entropy (or Gini impurity, more in a moment) of that child. The weighted average of the two child impurities is then compared to the parent impurity to produce an information gain score for that candidate threshold. This is pure counting arithmetic, no complex mathematics required.
4. **Select the best candidate**: the threshold that produces the highest information gain becomes the node's split point

This is the process Figure 7 above is demonstrating. The number of candidates is at most $n - 1$ where $n$ is the number of training examples, which is entirely manageable even for large datasets.

Figure 8 below demonstrates this. It replaces the scatter of Figure 7 with the **class distributions**, the smooth curves showing how apple and orange diameters are spread across the feature space. This paints a clearer picture of what is happening when we have a numerical feature, where we need to consider candidate thresholds for splitting the feature. The two distributions overlap in the middle: some apples are larger than some oranges, and no threshold can perfectly separate them. The shaded regions show which class the model would predict on each side of the current threshold; the darker the shading, the more that class dominates that region.

The small grey tick marks along the x-axis show every **candidate threshold** the algorithm evaluates, one midpoint between each pair of consecutive diameter values in the training data. The algorithm computes the information gain at each of these candidates in turn and selects the best one, i.e. the one that produces the biggest reduction in uncertainty. Drag the slider to any of these tick marks and watch the step-by-step calculation update: this is exactly the arithmetic the algorithm performs at each candidate position.

Notice that the best threshold, found by clicking **Snap to best IG**, sits in the region where the two distributions are most separated, not necessarily at their midpoint. The algorithm finds this position purely by maximising information gain, without ever looking at the distribution curves directly.

In [ ]:
# Listing 17.
%matplotlib widget
from visualisations.Figure_8 import show
show()

It is also worth noting that entropy and information gain are not the only options for finding the best splits. **Gini impurity** is another widely used measure of node purity that leads to the same split-selection process but replaces the logarithm in the entropy formula with a simpler squared term. There are also other criteria such as the **variance reduction** used in regression trees, where the target variable is continuous rather than a class label. The underlying idea is always the same: find the split that makes the resulting groups as homogeneous as possible, but different measures formalise that idea in slightly different ways.

#### Gini Impurity: An Alternative to Entropy

Scikit-learn's decision tree implementation uses a different measure by default called **Gini impurity**, and it is worth understanding what it is and why it is often preferred in practice.

Gini impurity measures the same underlying concept as entropy, how mixed the class labels are in a group, but uses a simpler formula:

$$G = 1 - \sum_{k=1}^{K} p_k^2$$

where $p_k$ is the proportion of examples belonging to class $k$. Reading this in plain English: *"Gini impurity is 1 minus the sum of the squared class proportions."*

The two special cases mirror those of entropy exactly:

- If all points belong to one class, say $p_\text{apple} = 1.0$, then $G = 1 - 1^2 = 0$. The group is perfectly pure.
- If the classes are split equally, $p_\text{apple} = p_\text{orange} = 0.5$, then $G = 1 - (0.5^2 + 0.5^2) = 0.5$. The group is maximally impure.

The split-selection process works identically to information gain. At each candidate threshold, the algorithm takes the current group of training examples at the node being evaluated (this is the **parent**) and splits it into two smaller groups called the **children**: one containing all examples with a feature value below the threshold, and one containing all examples at or above it. It computes the Gini impurity of each child group separately, reflecting how mixed the class labels are within each half. It then takes a weighted average of the two child impurities, weighting each child by the proportion of data points that fell into it, so a larger child group contributes more to the average than a smaller one. This weighted child impurity is subtracted from the Gini impurity of the parent group (the group before the split was made) to give a **Gini reduction** score for that candidate threshold. A large reduction means the split has meaningfully cleaned up the class labels; a small reduction means the children are barely purer than the parent was. The algorithm evaluates this score across every candidate threshold on every feature, and selects the split that produces the largest reduction. That split becomes the node's question, and the process then repeats recursively: each child node becomes the new parent for the next level of splitting, and the whole procedure runs again on the data that reached it, until the tree decides it has learned enough.

---

##### Why Use Gini Impurity Instead of Entropy?

The two measures are very similar in practice and almost always select the same split. The differences are subtle but worth knowing:

| | Entropy (Information Gain) | Gini Impurity |
|---|---|---|
| Formula | $-\sum p_k \log_2 p_k$ | $1 - \sum p_k^2$ |
| Computation | Requires a logarithm | Only requires squaring, faster |
| Range (two classes) | 0 to 1.0 bit | 0 to 0.5 |
| Sensitivity to rare classes | Slightly higher | Slightly lower |
| Default in scikit-learn | No | **Yes** |

The primary reason Gini impurity is preferred in scikit-learn and most production implementations is computational: squaring numbers is significantly faster than computing logarithms, which matters when the algorithm is evaluating thousands of candidate splits across large datasets.

The secondary reason is that entropy is slightly more sensitive to rare classes. This comes down to the mathematics of the two formulas: the logarithm in the entropy calculation grows very steeply as a probability approaches zero, meaning a class that appears only rarely in a group has an outsized influence on the entropy score. Gini impurity, by contrast, uses squaring, which shrinks small probabilities even further and gives rare classes less weight. In most balanced datasets this difference is negligible and the two measures will select exactly the same split. However, in heavily imbalanced datasets, where one class is far more common than the other, such as fraud detection where genuine fraud represents only a tiny fraction of transactions, entropy may prefer a split that isolates the rare class slightly better, while Gini may overlook it in favour of a split that improves purity for the dominant class. Neither behaviour is universally correct; it depends on whether correctly identifying the rare class matters more than overall accuracy. This is one of the reasons why class imbalance, covered in depth in Notebook 11, deserves careful attention before choosing both a model and its splitting criterion.

In practice, if you train a decision tree with `criterion='gini'` (the default) and then retrain with `criterion='entropy'`, the resulting trees are usually identical or differ only in one or two nodes near the leaves. 

The figure below applies Gini impurity to the same fruit dataset and candidate thresholds as Figure 8. The distribution plot is identical: drag the threshold through the overlapping apple and orange diameter distributions and watch the step-by-step panel update. The only difference is in the calculation: instead of logarithms, each step squares the class proportions and subtracts from 1. Compare the optimal threshold found here with the one found by information gain in Figure 8; they should agree, which is reassuring confirmation that both measures are capturing the same underlying idea.

In [ ]:
# Listing 18.
%matplotlib widget
from visualisations.Figure_9 import show
show()

---

The process then for constructing the tree is as follows.

1. Select the feature with the highest **information gain**, the feature that reduces uncertainty about the class label the most.
2. Search for the threshold $\theta$ on that feature that minimises the error (or equivalently, maximises the purity of the resulting partitions).
3. Create two child nodes and repeat until the partitions are pure or a stopping criterion is reached.

#### Decision Boundaries

Decision trees are **non-linear** models. This is a significant departure from the classifiers we have seen so far: logistic regression, for example, can only ever draw a single straight line (or flat hyperplane in higher dimensions) through the feature space. A decision tree has no such restriction.

Each split in a tree draws a single straight line through the feature space, but crucially, it is always **axis-aligned**: a vertical line if splitting on Feature 1, a horizontal line if splitting on Feature 2. On its own, one split produces a boundary no more expressive than a simple threshold. But as the tree grows deeper, each additional split subdivides the space further, carving it into a collection of rectangular regions, one per leaf node. The class label assigned to each region is simply whichever class dominates the training examples that fell into it.

The result is a **staircase-like boundary** that can approximate surprisingly complex shapes. Two splits might separate a circular cluster from its surroundings; three or four might handle a crescent, an L-shape, or interleaved classes that no straight line could ever separate. The more splits the tree makes, the finer the rectangular grid it carves, and the more closely it can conform to the true boundary between classes in the training data.

This flexibility is both a strength and a risk. A tree that keeps splitting until every leaf is perfectly pure will fit the training data exactly, but it will have carved the feature space into so many tiny rectangles that it has effectively memorised the training examples rather than learned a general rule. This is **overfitting**, and it is one of the central challenges in working with decision trees. Controlling how deep a tree is allowed to grow, through parameters like `max_depth`, `min_samples_split`, and `min_samples_leaf`, is the primary way of managing this tradeoff, and we will return to it shortly.

The figure below makes this process concrete. The scatter plot shows 60 fruit samples in two-dimensional feature space, diameter on the x-axis, colour score on the y-axis, with apples in blue and oranges in red. The two classes are deliberately interleaved so that no single straight line can separate them.

Each time you click **Add Next Split**, the algorithm finds the single split across both features that produces the greatest Gini reduction, adds it to the tree, and updates both panels simultaneously:

- **Scatter plot**: a new dashed boundary line appears, either vertical (a diameter split) or horizontal (a colour score split), clipped to the region it actually applies to. The shaded background shows the current predicted class for every point in the feature space, blue for Apple, red for Orange.
- **Tree diagram**: the corresponding node is split into two children, each labelled with its split question (or predicted class if a leaf), the count of apples and oranges that fell into it, and its Gini impurity score.

Watch how the shaded regions in the scatter become progressively smaller and purer with each split: the tree is carving the feature space into rectangular regions, trying to isolate each class. Also notice that the tree uses both features: some splits are vertical lines (diameter) and others are horizontal (colour score), because neither feature alone is sufficient.

The tree will stop growing at a maximum depth of 6. At that point the boundary is as refined as the depth limit allows, but notice that the regions are still not perfectly pure. This is the nature of overlapping data: no finite tree can achieve perfect classification, and attempting to do so by growing the tree indefinitely leads to overfitting, a topic we will return to.




In [ ]:
# Listing 19.
%matplotlib widget
from visualisations.Figure_10 import show
show()

Now let's see how this works, starting from a stump.

The two code cells below first search for the best single-threshold split (a single stump), then chain two stumps together to show how a second level of splitting allows the boundary to turn a corner.

### 6.2 Decision Tree with scikit-learn

Scikit-learn's `DecisionTreeClassifier` uses the **CART** (Classification and Regression Trees) algorithm. At each node, CART searches exhaustively across every feature and every candidate threshold, selecting the split that most reduces node impurity, measured either by Gini impurity (the default) or information entropy, both of which we have now covered.

The most important decisions when building a decision tree are not which splitting criterion to use, but **how much to let the tree grow**. An unconstrained tree will keep splitting until every leaf is pure, perfectly fitting the training data but generalising poorly to new examples. The following parameters are the primary tools for controlling this:

**Key hyperparameters:**

| Parameter | Default | What it controls |
|---|---|---|
| `criterion` | `'gini'` | The impurity measure used to evaluate splits. `'gini'` uses Gini impurity; `'entropy'` uses information gain. Both produce similar results; Gini is faster to compute. |
| `max_depth` | `None` | The maximum number of levels the tree is allowed to grow. The single most important lever for controlling overfitting. A shallow tree (depth 2-4) is often more generalisable than a deep one, even if it misclassifies some training examples. |
| `min_samples_split` | `2` | The minimum number of training examples a node must contain before it is allowed to split. Raising this prevents splits based on very small groups of examples, which are likely to be noise. |
| `min_samples_leaf` | `1` | The minimum number of training examples that must end up in each leaf after a split. Applied to the children rather than the parent, this prevents tiny leaf nodes representing only one or two unusual examples. |
| `max_features` | `None` | The number of features considered when searching for the best split at each node. Setting this below the total number of features introduces randomness, the foundation of the Random Forest algorithm covered later in this notebook. |
| `class_weight` | `None` | Adjusts the importance of each class during training. Setting `'balanced'` is useful when one class is much more common than the other, preventing the tree from simply predicting the majority class everywhere. |

These parameters interact: a large `max_depth` combined with a small `min_samples_leaf` will produce a very deep, complex tree that overfits. In practice, `max_depth` and `min_samples_leaf` are the two parameters most worth tuning first.

---

#### Training the model

Training a decision tree in scikit-learn follows the same three-step pattern used throughout this series of notebooks:

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# 1. Split the data into training and test sets
#    test_size=0.25 reserves 25% of the data for evaluation
#    stratify=y ensures all classes are proportionally represented in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

# 2. Create the model
#    max_depth=4 keeps the tree shallow enough to be interpretable
#    and prevents it from memorising the training data
clf = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=0)

# 3. Fit the model (this is where the learning happens)
#    CART searches for the best split at each node by evaluating every
#    feature and every candidate threshold, selecting the one that most
#    reduces Gini impurity. No gradient descent: one pass through the data.
clf.fit(X_train, y_train)
```

---

#### What the model has learned

After calling `fit()`, the trained model exposes several useful attributes:

```python
clf.get_depth()      # the actual depth reached by the fitted tree
clf.get_n_leaves()   # the number of leaf nodes (a measure of complexity)
clf.feature_importances_  # mean Gini reduction per feature across all splits
```

- `get_depth()` tells you how many levels the tree actually grew to, useful to check whether `max_depth` was the binding constraint or whether the tree stopped earlier due to pure leaves.
- `get_n_leaves()` is a direct measure of model complexity. More leaves means more rules, which means a higher risk of overfitting.
- `feature_importances_` gives the mean Gini impurity reduction contributed by each feature across all splits. A high score means that feature was consistently useful for making decisions throughout the tree.

The full rule set can be printed in human-readable form using `export_text`:

```python
from sklearn.tree import export_text

# export_text converts the fitted tree into a plain-text flowchart
# showing the exact feature and threshold checked at every node.
# Read it top to bottom, following the branches, to trace how the
# model classifies any new observation.
print(export_text(clf, feature_names=FEAT_NAMES))
```

---

#### Making predictions and getting probabilities

```python
# Predict the class label for each test example
#   Returns the class with the highest posterior probability for each row
y_pred = clf.predict(X_test)

# Predict the probability of each class for each test example
#   Returns a matrix of shape (n_samples, n_classes)
#   Each row sums to 1
y_proba = clf.predict_proba(X_test)
```

`predict()` passes each test example down the tree from root to leaf, following the branch that matches its feature values at each node, and returns the majority class at the leaf it reaches.

`predict_proba()` returns the proportion of training examples of each class that reached the same leaf, so a row of `[0.05, 0.91, 0.04]` means 91% of training examples that reached this leaf belonged to the second class.

---

#### Evaluating the model

```python
from sklearn.metrics import accuracy_score, classification_report

# Overall fraction of test examples classified correctly
accuracy = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {accuracy:.1%}')

# Detailed per-class breakdown of precision, recall, and F1 score
#   (these metrics are covered in depth in Notebooks 10 and 11)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))
```

---

#### A Real-World Inspired Example: Cloud Type Classification

Weather forecasting and atmospheric science rely heavily on automated classification of cloud types from sensor data. Different cloud types form under different atmospheric conditions, occur at different altitudes, and have very different optical properties, making them a natural candidate for a decision tree classifier. The rules a tree learns from this data are physically interpretable: *"if the temperature is below −30°C and the optical thickness is less than 2, predict Cirrus"* is exactly the kind of rule a meteorologist would recognise as valid.

The dataset introduced below is synthetic but physically grounded; the feature distributions are based on the known characteristics of each cloud type.

---

##### Dataset Description

Each row represents a single cloud observation, described by four continuous measurements. It is stored in the file `data/clouds.csv`:

```csv
altitude_km,vertical_extent_km,temperature_c,optical_thickness,cloud_type
2.152,2.758,6.032,4.96,Cumulus
1.48,2.211,8.541,5.442,Cumulus
...
```

| Feature | Units | Type | Description |
|---|---|---|---|
| `altitude_km` | km | Continuous | Height of the cloud base above sea level. Cirrus forms at 6-12 km; stratus and cumulus form below 3 km. |
| `vertical_extent_km` | km | Continuous | Thickness of the cloud layer from base to top. Deep cumulus can extend several kilometres; stratus and cirrus are typically thin. |
| `temperature_c` | °C | Continuous | Air temperature at cloud altitude. Temperature drops with altitude at roughly 6.5°C per km, so high-altitude cirrus clouds are extremely cold. |
| `optical_thickness` | 0-10 | Continuous | How opaque the cloud appears. Thick stratus layers block most sunlight (high score); wispy cirrus is nearly transparent (low score). |

The target variable is **cloud_type**, with three classes:

| Class | Typical characteristics |
|---|---|
| Cumulus | Low-to-mid altitude (approx. 2 km), moderate thickness, mild temperature, moderate opacity: the classic fair-weather puffy cloud |
| Stratus | Low altitude (approx. 1 km), thin layer, mild temperature, high opacity: the featureless grey overcast that blocks sunlight |
| Cirrus | Very high altitude (approx. 9 km), thin wispy layer, very cold (approx. −40°C), near-transparent: ice crystal clouds at the top of the troposphere |

1,800 observations are generated in total, 600 per class, with deliberate overlap between cumulus and stratus (both occur at low altitudes with mild temperatures) so the tree must use multiple features to separate them. The larger dataset gives the tree more stable split thresholds and makes the learned rules more physically meaningful.

---

##### What the Code Does

The code cell below works through the full workflow in four steps:

**1. Load the dataset.** The cloud observations saved to `data/clouds.csv` are read into a DataFrame. The `cloud_type` column is encoded as integers (Cumulus = 0, Stratus = 1, Cirrus = 2) for use with scikit-learn.

**2. Split into training and test sets.** 1,350 observations (75%) are used to fit the tree; 450 (25%) are held back for evaluation. The `stratify=y_cloud` argument ensures all three classes are proportionally represented in both sets.

**3. Fit a depth-4 decision tree.** `DecisionTreeClassifier(criterion='gini', max_depth=4)` is trained on the training set. The depth limit of 4 keeps the tree shallow enough to be interpretable and prevents it from memorising the training data.

**4. Evaluate and print the rule set.** `classification_report` gives per-class precision, recall, and F1-score on the held-out test set. `export_text` converts the fitted tree into a plain-text flowchart showing the exact feature and threshold checked at every node. Read it top to bottom, following the branches, and you can trace exactly how the model classifies any new observation.


In [ ]:
# Listing 20.
# ── Decision Tree on atmospheric data — cloud type classification ─────────────
#
# Loads the synthetic cloud observation dataset from data/clouds.csv and
# trains a depth-4 decision tree to classify cloud type from four physical
# measurements: altitude, vertical extent, temperature, and optical thickness.

# ── Load dataset ──────────────────────────────────────────────────────────────
# pd.read_csv reads the CSV into a DataFrame — rows are observations,
# columns are features plus the target label.
df = pd.read_csv('data/clouds.csv')

CLASS_NAMES = ['Cumulus', 'Stratus', 'Cirrus']
FEAT_NAMES  = ['altitude_km', 'vertical_extent_km',
               'temperature_c', 'optical_thickness']

# ── Separate features from labels ─────────────────────────────────────────────
# X contains the four input features; y contains the integer class label.
# pd.Categorical.from_codes maps the string class names to integers
# in the order defined by CLASS_NAMES.
X_cloud = df[FEAT_NAMES].values
y_cloud = pd.Categorical(df['cloud_type'], categories=CLASS_NAMES).codes

# ── Train / test split ────────────────────────────────────────────────────────
# stratify=y_cloud ensures each class is proportionally represented in both sets.
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cloud, y_cloud, test_size=0.25, random_state=0, stratify=y_cloud
)

# ── Fit a depth-4 decision tree ───────────────────────────────────────────────
# max_depth=4 keeps the tree interpretable — deep enough to capture the
# overlapping stratus/cumulus boundary, shallow enough to generalise.
clf = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=0)
clf.fit(X_tr, y_tr)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred = clf.predict(X_te)

print('Decision Tree — Cloud Type Classification')
print('=' * 50)
print(f'Training samples : {len(X_tr)}')
print(f'Test samples     : {len(X_te)}')
print(f'Tree depth       : {clf.get_depth()}')
print(f'Number of leaves : {clf.get_n_leaves()}')
print(f'Test accuracy    : {accuracy_score(y_te, y_pred):.1%}')
print()
print(classification_report(y_te, y_pred, target_names=CLASS_NAMES))

# ── Print the human-readable rule set ────────────────────────────────────────
# export_text converts the fitted tree into a plain-text flowchart showing
# exactly which feature and threshold is checked at each node.
print('Learned decision rules:')
print('-' * 50)
print(export_text(clf, feature_names=FEAT_NAMES))

The printed rule set from `export_text` above is precise but can be difficult to follow once the tree has more than a couple of levels. A visual representation makes the structure far easier to read: you can see at a glance how the tree branches, which features are used at each level, and how pure each node is.

The code below renders the fitted tree as a diagram. Each box is one node: internal nodes show the split condition, the Gini impurity, the number of training samples that reached it, and how those samples are distributed across the three classes. Leaf nodes show the final prediction. The shading tells you about purity: a darker, more saturated colour means the node is dominated by one class; a pale, washed-out colour means the samples are still mixed. Follow any path from the root at the top down to a leaf at the bottom and you have traced the exact sequence of questions the model asks to classify a new observation.


####  Overfitting: When a Tree Learns Too Much

So far we have trained a single tree at a fixed depth of 4. But how did we know that depth 4 was a good choice? And what would happen if we let the tree grow deeper?

Recall that a decision tree with no depth limit will keep splitting until every leaf contains only one class, a state of perfect purity. On the training data this looks impressive: the model correctly classifies every single example it was trained on, achieving 100% training accuracy. But this comes at a cost. To achieve perfect purity the tree has had to carve the feature space into an enormous number of tiny rectangular regions, many of which exist purely to accommodate individual unusual training examples: noise, measurement error, or just natural variation that does not reflect any real pattern in the data.

When a new, unseen observation arrives, it is unlikely to land in exactly the right tiny region. The tree has memorised the training set rather than learned a general rule. This is **overfitting**, and it is one of the most important failure modes in machine learning.

The figure below makes this concrete by training the same decision tree at every depth from 1 to 20 and recording two things at each depth:

- **Training accuracy**: how well the model fits the data it was trained on
- **Test accuracy**: how well it performs on the 25% of observations it has never seen

A well-generalising model should show similar scores on both. The **gap between the two curves**, the shaded region, is the overfitting gap. As depth increases, watch what happens: training accuracy climbs steadily toward 100%, while test accuracy peaks at a relatively shallow depth and then begins to plateau or even decline. The right panel shows why: the number of leaf nodes grows rapidly with depth, and each additional leaf is an additional rule that the model has fitted specifically to the training data.

The green dashed line marks the depth of 4 used in the previous cell. Notice where it sits relative to the peak of the test accuracy curve; this was a reasonable choice, but the figure tells you whether it was truly optimal for this dataset.

In [ ]:
# Listing 21.
# ── Visualise the fitted decision tree ───────────────────────────────────────
# plot_tree draws the full tree structure — each internal node shows the
# split condition, Gini impurity, sample count, and class distribution.
# Each leaf shows the majority class prediction.
fig, ax = plt.subplots(figsize=(12, 8))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

plot_tree(
    clf,
    feature_names=FEAT_NAMES,       # column names shown at each split
    class_names=CLASS_NAMES,         # class labels shown at each leaf
    filled=True,                     # colour each node by its majority class
    rounded=True,                    # rounded box corners
    fontsize=9,                      # readable at this figure size
    ax=ax,
)

ax.set_title(
    'Fitted Decision Tree — Cloud Type Classification  (max depth 4)',
    fontsize=12,
)
plt.tight_layout()
plt.show()


---

#### Overfitting and Underfitting in Decision Trees

The figure below illustrates one of the most fundamental tensions in machine learning: the tradeoff between a model that is **too simple** to capture the patterns in the data, and one that is **too complex** and captures noise as well as signal.

**Underfitting** occurs when the tree is too shallow to represent the true structure of the data. A tree of depth 1, a single stump, can only ask one question, and if the classes are not linearly separable by a single feature, the model will perform poorly on both training and test data. The signature of underfitting is low accuracy on both curves simultaneously. In the cloud classification problem, a very shallow tree cannot distinguish cumulus from stratus because both occur at low altitudes; it needs at least a second split on optical thickness or temperature to separate them.

**Overfitting** is the opposite problem and the greater risk with decision trees. An unconstrained tree will grow until every leaf is pure, creating a vast number of tiny rectangular regions in the feature space, one for every quirk, outlier, and measurement error in the training data. The model achieves near-perfect training accuracy but fails on new examples because those tiny regions are artefacts of the training data, not real patterns. The signature of overfitting is a large and growing gap between the training accuracy curve (climbing toward 100%) and the test accuracy curve (plateauing or declining).

In [ ]:
# Listing 22.
%matplotlib widget
from visualisations.Figure_11 import show
show()


---

The printed summary confirms what the curves show. The best test accuracy of 99.3% is achieved at a surprisingly shallow depth of just 3, the tree needs very few questions to classify most clouds correctly, which reflects how physically distinct the three cloud types are across the ten features in this dataset.

At maximum depth 20, training accuracy reaches 100%: the tree has memorised every training example perfectly, including the 5% of observations that were deliberately mislabelled. Yet test accuracy remains at 99.3%, identical to depth 3. The overfitting gap of just 0.7% is remarkably small for a depth-20 tree, and tells us two things: first, the dataset is large enough (4,500 training examples) that even a very deep tree struggles to overfit badly; second, the deliberately mislabelled examples are the primary source of irreducible error, since no model, however complex, can correctly classify an observation whose label is wrong by design.

The takeaway here is that `max_depth=3` or `max_depth=4` would be the sensible choice for this dataset. It achieves the same test performance as a depth-20 tree with a fraction of the complexity, training time, and memory cost, and produces a tree shallow enough to be printed and read as a set of physical rules.

---

##### Parameters That Control This Tradeoff

Several `DecisionTreeClassifier` parameters directly influence where the model sits on the underfitting-overfitting spectrum:

| Parameter | Effect on underfitting | Effect on overfitting |
|---|---|---|
| `max_depth` | Too small → underfits | Too large → overfits. The single most important control. |
| `min_samples_split` | Too large → underfits (nodes that could usefully split are stopped) | Too small → overfits (splits on tiny, noisy groups) |
| `min_samples_leaf` | Too large → underfits (forces large, impure leaves) | Too small → overfits (allows single-example leaves) |
| `max_features` | Too small → underfits (misses informative features) | Larger values → more thorough search, slightly higher overfit risk |
| `criterion` | Some effect on both | Marginal effect on either; both Gini and entropy produce similar depths |

In practice, `max_depth` and `min_samples_leaf` are the two most effective levers. A common starting point is `max_depth` between 3 and 8 and `min_samples_leaf` between 5 and 20, tuned by comparing training and test accuracy curves as shown in Figure 11.

Another symptom of overfitting in decision trees is **high variance**: small changes to the training data (adding a few examples, removing a few outliers) can dramatically alter the structure of the learned tree. A completely different set of splits may emerge from only a minor perturbation of the data. This instability is an inherent property of the greedy top-down splitting algorithm and is one of the motivations for ensemble methods like Random Forests, which average over many trees trained on slightly different subsets of the data to produce a more stable model.

---

##### When to Use Decision Trees

Decision trees are a strong choice when:

- **Interpretability matters**: the learned rules can be printed, read, and validated by a domain expert. A doctor, engineer, or regulator can follow the logic of a depth-4 tree in a way they cannot for a neural network.
- **Mixed feature types are present**: decision trees handle numerical and categorical features natively without requiring normalisation or one-hot encoding.
- **Non-linear relationships exist**: unlike logistic regression, a tree can capture complex, non-linear boundaries by combining multiple axis-aligned splits.
- **A fast baseline is needed**: trees train quickly and require minimal preprocessing, making them a useful first model before investing in more complex approaches.

##### When to Be Cautious

Decision trees are less suitable when:

- **Depth is not carefully controlled**: without `max_depth` or `min_samples_leaf` constraints, overfitting is almost guaranteed on any reasonably complex dataset.
- **Stability matters**: the high variance of individual trees means small data changes can produce very different models, which is problematic in production systems where consistency is important.
- **Classes are imbalanced**: a tree trained on imbalanced data tends to favour the majority class, creating large pure regions for the common class and ignoring the minority class entirely. Setting `class_weight='balanced'` mitigates this but does not eliminate it.
- **Smooth, continuous decision boundaries are needed**: because all splits are axis-aligned, a tree approximates diagonal or curved boundaries with a staircase pattern, which requires many splits to approximate accurately and increases overfit risk.



---

## 7. Random Forest

🔙 [Back to Table of Contents](#Table-of-Contents)

### 7.1 From Greedy Trees to Random Ensembles

A standard decision tree is **greedy**: at each node it always chooses the *best* available feature and the *best* split threshold, the one that maximises the Gini impurity reduction at that moment (or similar for information gain). This sounds sensible, but it has a significant weakness. The greedy algorithm only looks one step ahead. It cannot see that a mediocre split now might open up a much better split at the next level, and it cannot discover combinations of features that are only useful together.

Consider two features that are individually weak separators but together form a strong combined boundary, perhaps cloud altitude and optical thickness are each inconclusive on their own, but together they cleanly distinguish cirrus from stratus. A greedy tree might never discover this combination because neither feature looked particularly promising at the root node, so the algorithm commits to a different split early and never revisits the decision.

A deeper problem is that greedy trees are **unstable**. As we saw in the overfitting section, small changes to the training data can produce dramatically different tree structures, a property known as high variance. A model with high variance is sensitive to the particular sample of data it happened to be trained on, which makes it unreliable in practice.

---

#### The Ensemble Idea

The solution is not to build one better tree; it is to build many imperfect trees and combine their predictions. This is the core idea behind **ensemble methods**: individual models (called *weak learners*) each make different mistakes, but when their predictions are aggregated, the errors tend to cancel out and a stronger collective prediction emerges.

For this to work, the individual trees must make **different** mistakes: their errors must be *uncorrelated*. If every tree made the same mistakes (because they were all trained on the same data in the same way), aggregating them would not help. The key to making errors diverse and uncorrelated is **randomness**.

---

#### The Random Forest

The **Random Forest** introduces randomness at two levels during training:

**1. Bootstrap sampling.** Each tree in the forest is trained on a different random subset of the training data, drawn *with* replacement. This means some training examples appear multiple times in a tree's training set, and others do not appear at all. Because each tree sees a slightly different version of the data, each tree makes different mistakes. This resampling technique is called **bootstrapping**, and the data not used to train a given tree (roughly one third of the training set) is called the **out-of-bag (OOB)** sample. The OOB sample can be used as a free internal validation set without needing a separate held-out test set.

**2. Random feature selection.** At each node, instead of searching all available features for the best split, only a random subset of $m$ features is considered as split candidates. A common default is $m = \sqrt{p}$ where $p$ is the total number of features. This has two effects: it forces individual trees to explore different feature combinations, and it prevents any single dominant feature from controlling the structure of every tree in the forest. A feature that would always win the competition at the root node no longer has that opportunity in most trees, giving other features a chance to contribute.

Together these two sources of randomness ensure that the trees in the forest are diverse: each has a different structure, was trained on different data, and made different split decisions. A single randomly-built tree will perform worse than a carefully optimised single tree. But the ensemble of many such trees, combined by majority vote, reliably outperforms any individual tree.

---

#### Combining Predictions by Majority Vote

Once the forest is trained, classifying a new observation is straightforward: pass it through every tree in the forest, collect each tree's predicted class, and take the majority vote:

$$\hat{y} = \text{mode}\left(\hat{y}^{(1)}, \hat{y}^{(2)}, \ldots, \hat{y}^{(T)}\right)$$

where $T$ is the number of trees and $\hat{y}^{(t)}$ is the prediction of the $t$-th tree. The mode is simply the most frequently predicted class across all trees.

This aggregation step is sometimes called **bagging** (Bootstrap AGGregating), a general technique for reducing variance by averaging the predictions of models trained on bootstrapped samples. Random Forests extend bagging with the additional random feature selection step, which further decorrelates the trees and improves the ensemble's performance.

The central insight is this: **diverse errors cancel; correlated errors do not.** If trees make random, uncorrelated mistakes, those mistakes will be outvoted by the correct predictions from the other trees. But if the trees all make the same systematic mistake, because they are all responding to the same misleading pattern in the data, majority voting cannot fix it. This is why the randomness injected during training is not a weakness to be tolerated but a deliberate and essential design choice.

---

#### How Many Trees?

A natural question is: how many trees should the forest contain? In general, more trees means lower variance and more stable predictions, but the returns diminish quickly. Going from 10 trees to 100 trees produces a large improvement; going from 500 trees to 1,000 produces very little. The main cost of more trees is computational: training time and memory scale linearly with $T$. In practice, 100-500 trees is a reasonable default for most problems, and the code cell below illustrates how accuracy evolves as the forest grows.

### 7.2 Random Forests with scikit-learn

Scikit-learn's `RandomForestClassifier` implements the full Random Forest algorithm described above. The interface is almost identical to `DecisionTreeClassifier` (the same `fit`, `predict`, and `predict_proba` methods apply), but internally it builds an ensemble of trees rather than a single one.

---

#### Training the Model

Training a Random Forest in scikit-learn follows a similar pattern to training a tree or logistic regression, with one additional consideration: because the forest trains many trees internally, fitting can take noticeably longer than a single tree on large datasets. Setting `n_jobs=-1` uses all available CPU cores and parallelises the training across trees, a significant speedup with no effect on the result.

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 1. Split the data into training and test sets
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

# 2. Create the forest
#    n_estimators=200: train 200 trees, each on a different bootstrap sample
#    max_features='sqrt': consider sqrt(n_features) candidates at each split
#    oob_score=True: compute a free validation score using out-of-bag samples
#    n_jobs=-1: use all available CPU cores (trees train independently)
rf = RandomForestClassifier(
    n_estimators=200,
    max_features='sqrt',
    oob_score=True,
    n_jobs=-1,
    random_state=0,
)

# 3. Fit: trains all 200 trees, each on a bootstrap sample of the training data
rf.fit(X_tr, y_tr)
```

---

#### What the Model Has Learned

After calling `fit()`, the trained model exposes several useful attributes:

```python
rf.estimators_          # list of the individual fitted DecisionTreeClassifier objects
rf.feature_importances_ # mean Gini reduction per feature, averaged across all trees
rf.oob_score_           # out-of-bag accuracy (a free internal validation score)
rf.classes_             # the class labels the model was trained on
rf.n_features_in_       # the number of features the model expects at prediction time
```

- `feature_importances_` gives the mean Gini impurity reduction contributed by each feature, averaged across all 200 trees and all bootstrap samples. This is a far more stable measure of feature relevance than the root split of a single tree, which depends heavily on the particular sample drawn.
- `oob_score_` is the accuracy computed on the out-of-bag samples, the roughly one third of training examples not used by each tree. It is a free estimate of generalisation performance and should be close to the test accuracy. A large gap between the two is a warning sign.

| Attribute | What it contains |
|---|---|
| `rf.estimators_` | A list of the individual fitted `DecisionTreeClassifier` objects |
| `rf.feature_importances_` | Mean Gini impurity reduction per feature, averaged across all trees |
| `rf.oob_score_` | Out-of-bag accuracy, only available if `oob_score=True` was set |
| `rf.classes_` | The class labels the model was trained on |
| `rf.n_features_in_` | The number of features the model expects at prediction time |

---

#### Making Predictions and Getting Probabilities

```python
# Predict the class label for each test example
#   Each tree votes for a class; the majority vote wins
y_pred = rf.predict(X_te)

# Predict the probability of each class for each test example
#   Returns a matrix of shape (n_samples, n_classes)
#   Each row sums to 1 (averaged across all trees)
y_proba = rf.predict_proba(X_te)
```

`predict_proba()` averages the class probabilities across all 200 trees. Because many trees contribute to each prediction, these averaged probabilities are generally better calibrated and more stable than those from a single tree.

---

#### Evaluating the Model

```python
from sklearn.metrics import accuracy_score, classification_report

print(f'OOB accuracy  : {rf.oob_score_:.1%}')
print(f'Test accuracy : {accuracy_score(y_te, y_pred):.1%}')
print()
print(classification_report(y_te, y_pred, target_names=CLASS_NAMES))
```

Always check whether the OOB accuracy and test accuracy are in close agreement. A large gap suggests the OOB estimate is unreliable, which can happen with very small datasets or heavily imbalanced classes.

---

#### Key Hyperparameters

| Parameter | Default | What it controls |
|---|---|---|
| `n_estimators` | `100` | Number of trees in the forest. More trees = lower variance but slower training. Returns diminish beyond ~200-500. |
| `max_depth` | `None` | Maximum depth of each individual tree. Unlike a single tree, deep trees in a forest do not necessarily overfit because the ensemble averaging absorbs variance. |
| `max_features` | `'sqrt'` | Number of features considered at each split. `'sqrt'` uses $\sqrt{p}$ features; `'log2'` uses $\log_2 p$; an integer or float sets an exact count or fraction. Lower values create more diverse, decorrelated trees. |
| `min_samples_split` | `2` | Minimum samples required to split a node. Raising this reduces complexity of individual trees. |
| `min_samples_leaf` | `1` | Minimum samples required in each leaf. Values of 5-20 often improve generalisation. |
| `bootstrap` | `True` | Whether to use bootstrap sampling. Setting `False` removes the bagging benefit and is rarely recommended. |
| `oob_score` | `False` | If `True`, computes a validation accuracy score using out-of-bag samples after training. |
| `n_jobs` | `1` | Number of CPU cores to use. `-1` uses all available cores: trees are independent so this dramatically speeds up training on large datasets. |
| `criterion` | `'gini'` | Impurity measure: `'gini'` or `'entropy'`, exactly as in `DecisionTreeClassifier`. |
| `class_weight` | `None` | Per-class weights. `'balanced'` adjusts weights inversely proportional to class frequencies, useful for imbalanced datasets. |
| `random_state` | `None` | Seed for reproducibility. Always set this when comparing models or reporting results. |

---

#### A Note on Hyperparameter Tuning

The most impactful parameters to tune first are `n_estimators`, `max_features`, and `min_samples_leaf`. A practical starting strategy:

- Set `n_estimators` to at least 100 and increase if variance in the OOB score is high
- Try `max_features` values of `'sqrt'`, `'log2'`, and `0.5` (half the features)
- Experiment with `min_samples_leaf` values of 1, 5, 10, and 20

`max_depth` is less critical than in a single tree: individual trees in a forest can be deep without the ensemble overfitting badly, because the averaging step absorbs much of the variance. That said, limiting depth can speed up training significantly on large datasets.

---

##### Dataset Description

We'll shortly train a random forest classifier, but first, let's learn about the data we'll use to do this. Take a look at the data below, which is a brief extract from the `clouds_complex.csv` dataset. Each row in the file represents a single cloud observation described by ten continuous measurements.

```csv
altitude_km,vertical_extent_km,temperature_c,optical_thickness,droplet_radius_um,liquid_water_path,wind_shear,humidity_pct,albedo,pressure_hpa,cloud_type
2.341,3.102,7.841,4.823,13.241,312.443,3.821,71.203,0.724,921.443,Cumulus
0.921,0.812,8.203,8.102,8.441,98.321,4.102,87.341,0.831,981.203,Stratus
...
```

The table below explains the variables in the dataset.

| Feature | Units | Type | Description |
|---|---|---|---|
| `altitude_km` | km | Continuous | Height of the cloud base above sea level. Cirrus forms at 6-12 km; stratus and cumulus form below 3 km. |
| `vertical_extent_km` | km | Continuous | Thickness of the cloud layer from base to top. Deep cumulus can extend several kilometres; stratus and cirrus are typically thin. |
| `temperature_c` | °C | Continuous | Air temperature at cloud altitude. Temperature drops with altitude at roughly 6.5°C per km, so high-altitude cirrus is extremely cold. |
| `optical_thickness` | 0-12 | Continuous | How opaque the cloud appears to incoming sunlight. Thick stratus is highly opaque; wispy cirrus is near-transparent. |
| `droplet_radius_um` | μm | Continuous | Effective radius of water droplets or ice crystals. Has a non-linear U-shaped relationship with altitude: smallest near the surface and at very high altitudes, largest in mid-level clouds. |
| `liquid_water_path` | g/m² | Continuous | Total mass of liquid water in a vertical column through the cloud. Interacts non-linearly with vertical extent: thicker clouds hold disproportionately more water. |
| `wind_shear` | m/s/km | Continuous | Change in wind speed across the cloud layer. U-shaped with altitude: higher near the surface and at the top of the troposphere. |
| `humidity_pct` | % | Continuous | Relative humidity at cloud altitude. Stratus forms in very humid conditions (approx. 88%); cirrus forms in much drier air (approx. 35%). |
| `albedo` | 0-1 | Continuous | Fraction of incoming sunlight reflected back to space. A non-linear function of optical thickness and liquid water path: dense stratus reflects most sunlight; thin cirrus reflects very little. |
| `pressure_hpa` | hPa | Continuous | Atmospheric pressure at the cloud base. Decreases approximately exponentially with altitude, providing an indirect but physically meaningful proxy for height. |

The target variable is **cloud_type**, with three classes:

| Class | Typical characteristics |
|---|---|
| Cumulus | Low-to-mid altitude (approx. 2.2 km), well-developed vertical extent, moderate optical thickness, moderate humidity: the classic fair-weather puffy cloud |
| Stratus | Low altitude (approx. 1.0 km), thin layer, high optical thickness, high humidity (approx. 88%): the featureless grey overcast that blocks sunlight |
| Cirrus | Very high altitude (approx. 9.5 km), thin wispy layer, very cold (approx. −42°C), near-transparent, low humidity (approx. 35%): ice crystal clouds at the top of the troposphere |

6,000 observations are generated in total, 2,000 per class, with several sources of difficulty that make this a more realistic and challenging classification problem than `clouds.csv`:

- **Wider class distributions**: larger standard deviations on altitude and temperature mean cumulus and stratus overlap substantially at the low-altitude end of the feature space
- **Non-linear feature relationships**: droplet radius, liquid water path, albedo, and wind shear all have non-linear relationships with altitude and cloud type that a single decision tree may struggle to capture
- **Correlated noise**: measurement noise is correlated across features, simulating the kind of smooth sensor variation seen in real atmospheric data
- **Label noise**: approximately 5% of observations are deliberately mislabelled to simulate ambiguous real-world cases where expert meteorologists might disagree on the cloud type

Together these properties mean that no simple set of threshold rules can achieve perfect classification, making this dataset well suited to illustrate the advantage of an ensemble approach over a single decision tree.

#### Comparing a Decision Tree and a Random Forest

The code cell below trains both a single decision tree and a random forest on the complex cloud dataset and compares their performance side by side. Using the same train/test split for both models ensures the comparison is fair: any difference in the results is due to the model itself, not luck in the data split.

The single decision tree is given a generous depth of 6 and a `min_samples_leaf` of 5 to give it the best realistic chance of capturing the non-linear relationships in the data while limiting overfitting. The random forest uses 200 trees, each trained on a bootstrap sample with approximately $\sqrt{10} \approx 3$ features considered at each split.

A few things to pay attention to in the output:

**OOB accuracy**: the random forest reports an **out-of-bag accuracy** score in addition to the test accuracy. This is computed using the training examples that were not used to train each individual tree, roughly one third of the data per tree, which the tree has never seen during its own training, making them a natural held-out validation set that comes for free without requiring a separate split. It is a free internal estimate of generalisation performance and should be close to the test accuracy. If the two are very different, something unusual is happening with the data or the model.

**Per-class performance**: look at the precision and recall for Cumulus and Stratus in particular. These two classes overlap in altitude and temperature, and the label noise in the dataset makes them harder to separate. The forest's ensemble averaging should produce noticeably better scores for these classes than the single tree.

**Feature importances**: at the end of the output, the forest reports how much each of the ten features contributed to reducing Gini impurity across all 200 trees. Features with high importance were consistently useful across many trees and many bootstrap samples, a more reliable signal than the feature chosen at the root node of a single tree, which is heavily influenced by whichever sample happened to be in the training set.


In [ ]:
# Listing 23.
# ── Random Forest vs Decision Tree — complex cloud classification ─────────────
#
# We train both a single decision tree and a random forest on the complex
# cloud dataset and compare their performance. The forest should outperform
# the single tree — particularly on precision and recall for the overlapping
# cumulus/stratus classes — because the ensemble averaging reduces the
# variance introduced by the non-linear features and label noise.

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('data/clouds_complex.csv')

CLASS_NAMES = ['Cumulus', 'Stratus', 'Cirrus']
FEAT_NAMES  = [
    'altitude_km', 'vertical_extent_km', 'temperature_c',
    'optical_thickness', 'droplet_radius_um', 'liquid_water_path',
    'wind_shear', 'humidity_pct', 'albedo', 'pressure_hpa',
]

X = df[FEAT_NAMES].values
y = pd.Categorical(df['cloud_type'], categories=CLASS_NAMES).codes

# ── Train / test split ────────────────────────────────────────────────────────
# stratify=y ensures each class is proportionally represented in both sets.
# Using the same split for both models guarantees a fair comparison — any
# difference in performance is due to the model, not the data split.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

print(f'Training samples : {len(X_tr)}  ({len(X_tr)//3} per class)')
print(f'Test samples     : {len(X_te)}')
print()

# ════════════════════════════════════════════════════════════════════════════
# MODEL 1 — Single Decision Tree
# We use max_depth=6 to give the tree enough room to capture the non-linear
# relationships in the data while limiting overfitting.
# ════════════════════════════════════════════════════════════════════════════
dt = DecisionTreeClassifier(
    criterion='gini',
    max_depth=6,
    min_samples_leaf=5,  # prevents tiny leaf nodes on noisy examples
    random_state=0,
)
dt.fit(X_tr, y_tr)
y_pred_dt = dt.predict(X_te)

print('=' * 55)
print('Single Decision Tree  (max_depth=6, min_samples_leaf=5)')
print('=' * 55)
print(f'Tree depth       : {dt.get_depth()}')
print(f'Number of leaves : {dt.get_n_leaves()}')
print(f'Test accuracy    : {accuracy_score(y_te, y_pred_dt):.1%}')
print()
print(classification_report(y_te, y_pred_dt, target_names=CLASS_NAMES))

# ════════════════════════════════════════════════════════════════════════════
# MODEL 2 — Random Forest
# 200 trees, each trained on a bootstrap sample of the training data with
# sqrt(n_features) = sqrt(10) ≈ 3 features considered at each split.
# oob_score=True gives us a free internal validation score using the
# out-of-bag samples — the training examples not used by each tree.
# ════════════════════════════════════════════════════════════════════════════
rf = RandomForestClassifier(
    n_estimators=200,       # 200 trees in the forest
    max_features='sqrt',    # sqrt(10) ≈ 3 features per split
    min_samples_leaf=3,     # slightly smaller than the single tree —
                            # the ensemble averaging compensates for variance
    oob_score=True,         # compute out-of-bag accuracy as a free validation score
    n_jobs=-1,              # use all available CPU cores — trees are independent
    random_state=0,
)
rf.fit(X_tr, y_tr)
y_pred_rf = rf.predict(X_te)

print('=' * 55)
print('Random Forest  (200 trees, max_features=sqrt)')
print('=' * 55)
print(f'OOB accuracy     : {rf.oob_score_:.1%}')
print(f'Test accuracy    : {accuracy_score(y_te, y_pred_rf):.1%}')
print()
print(classification_report(y_te, y_pred_rf, target_names=CLASS_NAMES))

# ════════════════════════════════════════════════════════════════════════════
# COMPARISON SUMMARY
# ════════════════════════════════════════════════════════════════════════════
dt_acc = accuracy_score(y_te, y_pred_dt)
rf_acc = accuracy_score(y_te, y_pred_rf)

print('=' * 55)
print('Summary')
print('=' * 55)
print(f'Decision tree accuracy : {dt_acc:.1%}')
print(f'Random forest accuracy : {rf_acc:.1%}')
print(f'Improvement            : {(rf_acc - dt_acc):.1%} percentage points')
print()
print('Feature importances (Random Forest):')
print('-' * 40)
importances = rf.feature_importances_
order       = np.argsort(importances)[::-1]
for rank, i in enumerate(order, 1):
    bar = '█' * int(importances[i] * 60)
    print(f'  {rank:2d}. {FEAT_NAMES[i]:<24}  {importances[i]:.4f}  {bar}')


---

#### Interpreting the Results

**Data split**

4,500 observations were used for training (1,500 per class) and 1,500 were held back for testing. The balanced class distribution means accuracy is a fair summary metric here: there is no majority class artificially inflating the headline number.

---

**Single Decision Tree**

The tree grew to its maximum permitted depth of 6, producing 34 leaf nodes. Each leaf node is one decision rule, so the entire model's logic can be expressed in 34 statements, which is one of the key practical strengths of a decision tree.

Test accuracy of 94.3% is a strong result for a single model on a dataset with deliberate label noise and overlapping classes. All three classes achieve similar precision and recall (0.93-0.96), which suggests the tree is not systematically favouring any one class. Cumulus and Stratus, the two most similar classes, both score 0.94 on both metrics, indicating the tree has found useful splits to distinguish them despite their overlapping altitude and temperature distributions.

---

**Random Forest**

The forest of 200 trees achieves 95.0% test accuracy, a modest but consistent improvement of 0.7 percentage points over the single tree. More telling is that the OOB accuracy of 94.8% is very close to the test accuracy of 95.0%. This agreement is reassuring: it suggests the model is not overfitting and that the OOB estimate is a reliable proxy for true generalisation performance on this dataset.

The per-class improvements are small but consistent across all three classes. Cirrus in particular improves from 0.94 to 0.96 recall: the forest is slightly better at catching the difficult edge cases at the boundary between high-altitude cumulus and cirrus.

The improvement may seem modest, but recall that the single tree was already well-tuned with `max_depth=6` and `min_samples_leaf=5`. The forest's advantage is most pronounced when the single tree is allowed to overfit; try removing the depth limit from the tree and rerunning to see the gap widen significantly.

---

**Feature importances**

The importances reveal which features the forest found most consistently useful across all 200 trees and all bootstrap samples:

- **Liquid water path** (0.211) is the single most informative feature: the total water content of the cloud column is highly characteristic of each cloud type. Stratus holds moderate liquid water in a thin layer; cumulus holds large amounts in a deep layer; cirrus holds almost none (it is made of ice crystals, not liquid droplets).
- **Pressure** (0.157) is a strong indirect proxy for altitude, since atmospheric pressure decreases approximately exponentially with height, it contains almost the same information as altitude but measured more precisely by instruments.
- **Droplet radius** (0.140) and **altitude** (0.126) are jointly important: their non-linear relationship with cloud type means both are needed together to resolve ambiguous cases.
- **Temperature** (0.124) provides useful corroborating information but is somewhat redundant with altitude and pressure, which is reflected in its slightly lower importance score.
- **Wind shear** (0.012) and **albedo** (0.010) contribute almost nothing. This is somewhat surprising for albedo, since it is a physically meaningful quantity, but in this dataset its value is largely determined by optical thickness and liquid water path, both of which the forest can already access directly. When two features carry the same information, the forest tends to concentrate importance on one and discount the other.

If you were building a real cloud classification system and needed to reduce the number of sensors, this importance ranking suggests you could drop wind shear and albedo with minimal loss of accuracy, while liquid water path and pressure should be measured as precisely as possible.


The two plots below help interpret what the random forest has actually learned, not just *how well* it performs, but *how* it makes its decisions and *where* it goes wrong.

The **left panel** shows the feature importances: how much each of the ten features contributed to reducing Gini impurity across all 200 trees and all bootstrap samples. Unlike the feature chosen at the root of a single tree, which depends heavily on the particular training sample, these importances are averaged across thousands of splits and are therefore a much more stable and trustworthy measure of each feature's predictive value.

The **right panel** shows the confusion matrix: a breakdown of predictions by true class. Each row corresponds to a true class; each column to a predicted class. The diagonal cells show correct predictions; the off-diagonal cells show misclassifications. Both the raw count and the row-normalised percentage are shown, so you can see not just how many errors were made but what fraction of each class they represent.

In [ ]:
# Listing 24.
# ── Random Forest — feature importance plot and confusion matrix ──────────────
#
# Two visualisations that help interpret what the random forest has learned:
#
#   Left  — Feature importances: the mean Gini impurity reduction contributed
#            by each feature across all 200 trees. A high score means that
#            feature was consistently useful for making splits across the forest.
#
#   Right — Confusion matrix: shows exactly which classes the model confuses
#            with which. Rows are true labels; columns are predicted labels.
#            A perfect classifier would have non-zero values only on the diagonal.

fig, (ax_imp, ax_cm) = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

# ════════════════════════════════════════════════════════════════════════════
# LEFT PANEL — Feature importances
# ════════════════════════════════════════════════════════════════════════════

# Sort features by importance descending so the most useful feature
# appears at the top of the bar chart — easier to read than an arbitrary order.
importances = rf.feature_importances_
order       = np.argsort(importances)          # ascending — for horizontal bar
feat_sorted = [FEAT_NAMES[i] for i in order]
imp_sorted  = importances[order]

# Colour bars by importance: darker = more important
colours = plt.cm.Blues(
    0.3 + 0.7 * imp_sorted / imp_sorted.max()
)

bars = ax_imp.barh(feat_sorted, imp_sorted, color=colours, edgecolor='white',
                   linewidth=0.5)

# Annotate each bar with its numeric importance score
for bar, val in zip(bars, imp_sorted):
    ax_imp.text(
        val + 0.001, bar.get_y() + bar.get_height() / 2,
        f'{val:.4f}',
        va='center', ha='left', fontsize=8.5, color='#333',
    )

ax_imp.set_xlabel('Mean Gini impurity reduction', fontsize=11)
ax_imp.set_title(
    'Feature importances — Random Forest (200 trees)\n'
    'How much each feature contributed to reducing impurity across all splits',
    fontsize=10,
)
ax_imp.set_xlim(0, imp_sorted.max() * 1.18)
ax_imp.grid(True, axis='x', alpha=0.2)
ax_imp.spines['top'].set_visible(False)
ax_imp.spines['right'].set_visible(False)

# ════════════════════════════════════════════════════════════════════════════
# RIGHT PANEL — Confusion matrix
# ════════════════════════════════════════════════════════════════════════════

# ConfusionMatrixDisplay wraps sklearn's confusion_matrix and draws it as
# a colour-coded grid. Each cell (i, j) shows how many observations with
# true label i were predicted as label j.
# normalize='true' shows row-wise proportions (fraction of each true class
# predicted as each predicted class) rather than raw counts — this makes
# it easier to compare performance across classes of different sizes.

cm = confusion_matrix(y_te, y_pred_rf)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES,
)
disp.plot(
    ax=ax_cm,
    colorbar=False,
    cmap='Blues',
)

# Add row-normalised percentages as a second annotation below the counts
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax_cm.text(
            j, i + 0.28,
            f'({cm_norm[i, j]:.1%})',
            ha='center', va='center',
            fontsize=8, color='white' if cm_norm[i, j] > 0.6 else '#333',
        )

ax_cm.set_title(
    'Confusion matrix — Random Forest\n'
    'Rows = true class   |   Columns = predicted class',
    fontsize=10,
)
ax_cm.set_xlabel('Predicted class', fontsize=11)
ax_cm.set_ylabel('True class', fontsize=11)

plt.suptitle(
    'Figure 12: Random Forest interpretation — '
    'feature importances and confusion matrix',
    fontsize=11,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

---

Feature importances are one of the most practically useful outputs of a Random Forest, and one of its key advantages over a single decision tree. A single tree's root split is heavily influenced by whichever training sample happened to be drawn, making it an unstable indicator of true feature relevance. The forest's importances, averaged across all 200 trees and all bootstrap samples, are far more reliable. High importance means a feature was consistently useful for making splits across many different subsets of the data, a much stronger signal. This connects directly to the discriminative feature selection ideas introduced in Notebook 3, but here the ranking is computed automatically from the training process rather than requiring a separate analysis step.


#### Different Ways to Make a Final Decision

So far, we've discussed forests that use the **majority vote** rule to make a decision (the mode): each tree casts one vote for its predicted class, and the class with the most votes wins. This is the default behaviour of `rf.predict()` and works well in most situations.

However, scikit-learn also exposes `rf.predict_proba()`, which returns the **averaged class probabilities** across all trees rather than a single hard label. Each tree produces a probability estimate for each class (based on the proportion of training examples of each class in its leaf node), and these are averaged across all 200 trees. This soft aggregation is often more informative than a hard vote:

- **More calibrated predictions**: instead of "this is Cirrus", you get "this is Cirrus with 94% probability, Cumulus with 5%, Stratus with 1%". A prediction with 51% vs 49% tells you something very different about uncertainty than one with 99% vs 0.5%.
- **Better threshold control**: in problems where the cost of different errors is unequal (for example, missing a rare but dangerous cloud formation matters more than a false alarm), you can adjust the decision threshold on the probability output rather than accepting the default majority vote.
- **Useful for downstream tasks**: if the cloud classifier feeds into a larger forecasting system, passing probabilities rather than hard labels preserves uncertainty information that the downstream model can use.

A third option is **weighted voting**, where trees with better OOB performance on the training data are given more influence over the final prediction. Scikit-learn does not implement this directly, but it can be approximated manually using the `estimators_` attribute and each tree's individual OOB score. In practice the improvement over simple averaging is usually small: the diversity of the trees matters far more than weighting them.

The choice between hard voting and soft probability averaging rarely changes the final predicted class for clear-cut examples, but makes a meaningful difference near decision boundaries, exactly the region where Cumulus and Stratus overlap in this dataset.

---

#### When to Use a Random Forest

- **Strong out-of-the-box performance**: forests typically outperform single trees with minimal hyperparameter tuning. The default settings (`n_estimators=100`, `max_features='sqrt'`) are a reasonable starting point for most problems.
- **Reliable feature importances**: the averaged importances are stable across runs and provide genuine insight into which features drive predictions.
- **High overfitting risk**: when a single tree overfits badly, a forest almost always reduces that variance substantially through ensemble averaging.
- **Medium to large datasets**: more data means more diverse bootstrap samples, which makes the individual trees less correlated and the ensemble more effective.

#### When to Be Cautious

- **Interpretability is essential**: a single decision tree can be printed and read by a domain expert; a forest of 200 trees cannot. If you need to explain exactly why a particular prediction was made, a forest is the wrong tool.
- **Memory constraints**: storing hundreds of fitted trees is expensive. On very large datasets with many features, a forest can consume substantial memory, and prediction time scales linearly with the number of trees.
- **Real-time prediction with tight latency requirements**: passing a single observation through 200 trees sequentially adds latency. For applications where predictions must be returned in microseconds, simpler models are preferable.
- **Highly imbalanced classes**: like single trees, forests can struggle with severe class imbalance unless `class_weight='balanced'` is set or the data is resampled before training.



---

## 8. Support Vector Machines

🔙 [Back to Table of Contents](#Table-of-Contents)

Support Vector Machines (SVMs) are one of the most mathematically elegant classifiers in machine learning. Like decision trees, they are **discriminative**: rather than modelling what each class looks like, they focus entirely on finding the best possible boundary between classes. But where a decision tree draws axis-aligned rectangular boundaries, an SVM draws a single, smooth, geometrically optimal boundary. It uses a precise definition of what "optimal" means.


### 8.1 The Maximum Margin Hyperplane

#### What Is a Hyperplane?

In two dimensions, the feature space is a flat 2-d plane, and every possible combination of two feature values corresponds to a point somewhere on it. A straight line drawn across that plane divides it into two half-planes. In three dimensions, the feature space is a volume, and a flat two-dimensional plane slicing through it divides the space into two half-spaces. In $d$ dimensions, the feature space has $d$ axes, one per feature, and the equivalent dividing object is called a **hyperplane**: a flat $(d-1)$-dimensional surface that cuts the feature space into two regions, one per class. The word "hyper" simply means we are generalising the familiar concept of a line or plane to spaces with more dimensions than we can easily visualise.

A hyperplane is defined by two quantities:

- $\mathbf{w}$, the **weight vector**: a list of numbers, one per feature, that together define the orientation of the hyperplane. Think of it as an arrow pointing at a right angle to the hyperplane's boundary surface (perpendicular to it), in the direction of the positive class. In a two-feature problem $\mathbf{w} = [w_1, w_2]$, where $w_1$ controls how much Feature 1 influences the boundary and $w_2$ controls how much Feature 2 does. A large value of $w_1$ means the boundary is steeply angled relative to the Feature 1 axis; a value of zero means Feature 1 plays no role at all.

- $b$, a **bias** (also called the intercept), which shifts the hyperplane away from the origin. Without $b$, every possible boundary would be forced to pass through the origin, an unnecessarily restrictive constraint.

Together, $\mathbf{w}$ and $b$ fully describe the position and orientation of the boundary in feature space. The hyperplane itself is the set of all points $\mathbf{x}$ where the following expression equals exactly zero:

$$\mathbf{w}^\top \mathbf{x} + b = 0$$

The notation $\mathbf{w}^\top \mathbf{x}$ means the dot product of $\mathbf{w}$ and $\mathbf{x}$: multiply each feature value by its corresponding weight and sum the results. The result, added to $b$, gives a single number called the **score** of a point. Points where this score equals zero sit exactly on the boundary; they are neither on one side nor the other. Points where the score is positive sit on one side of the boundary; points where it is negative sit on the other. The hyperplane is therefore the dividing surface where the score transitions from positive to negative, the decision boundary that separates the two classes.

To see why this makes sense, think of it geometrically. If you are standing exactly on the boundary line, the score is zero. Hence why the hyperplane itself is given by $\mathbf{w}^\top \mathbf{x} + b = 0$. Take one step toward the positive class and the score becomes positive. Take one step the other way and it becomes negative. The boundary is simply the set of all positions where the score is exactly balanced, neither positive nor negative.

Classification is then straightforward: compute the score for a new data point and check its sign:

$$\hat{y} = \begin{cases} +1 & \text{if } \mathbf{w}^\top \mathbf{x} + b \geq 0 \\ -1 & \text{if } \mathbf{w}^\top \mathbf{x} + b < 0 \end{cases}$$

Many hyperplanes could separate two classes, infinitely many, in fact, for any linearly separable dataset. The SVM's key insight is that not all of them are equally good. This leads naturally to the question of *learning*. Suppose we have a training set of labelled examples, where each point $\mathbf{x}_i$ has a known class label $y_i \in \{+1, -1\}$. We want to find values of $\mathbf{w}$ and $b$ such that the hyperplane $\mathbf{w}^\top \mathbf{x} + b = 0$ correctly separates all the training points, that is, every Class +1 point scores positive and every Class −1 point scores negative. Learning, in this context, means searching through the space of all possible $\mathbf{w}$ and $b$ values until we find a combination that produces the correct prediction for every training example. For a linearly separable dataset, infinitely many such combinations exist; any line that sits in the gap between the two classes will do. This is precisely the problem the SVM solves in a principled way: rather than picking any valid $\mathbf{w}$ and $b$, it picks the *best* one, the one that maximises the margin between the classes.

----

The interactive figure below lets you explore these ideas directly. The two sliders $w_1$ and $w_2$ control the orientation of the boundary: watch the red arrow (the weight vector $\mathbf{w}$) rotate as you change them, always staying perpendicular to the boundary line. The $b$ slider slides the boundary left or right without rotating it. The query point sliders let you place a point anywhere in the feature space by controlling its feature values. The point is shown as a star on the plot, and the prediction box below the controls shows the full score calculation and the resulting class prediction.

**A few things to try:**

- Set $w_2 = 0$ and drag $w_1$: the boundary becomes vertical and $\mathbf{w}$ points horizontally. Feature 2 has no influence on the prediction at all.
- Set $w_1 = w_2 = 1$: the boundary runs diagonally at 45° and $\mathbf{w}$ points toward the upper right.
- Keep $w_1$ and $w_2$ fixed and drag $b$: the boundary slides without rotating. This is what the bias term does: it decouples the position of the boundary from its orientation.
- Place the query point on one side of the boundary and watch the score be positive; drag it to the other side and watch it go negative. The boundary itself is exactly where the score equals zero.

---


In [ ]:
# Listing 25.
%matplotlib widget
from visualisations.Figure_13a import show
show()


#### What is $\|\mathbf{w}\|$?

The equation panel in the figure above shows a quantity written as $\|\mathbf{w}\|$. Before explaining what it means, it helps to think about what $\mathbf{w}$ actually represents in a practical sense.

Recall that $\mathbf{w} = [w_1, w_2]$ is a list of two numbers, one per feature. Each number says how strongly that feature pulls the decision boundary in a particular direction. If $w_1$ is large and $w_2$ is small, the boundary is mostly controlled by Feature 1; if both are equal, both features contribute equally. Together, $w_1$ and $w_2$ define both the *direction* the boundary faces and how *strongly* it is oriented that way.

$\|\mathbf{w}\|$, read as "the norm of $\mathbf{w}$", or simply the **length** of $\mathbf{w}$, captures just the second part of that: how strongly. It is computed as:

$$\|\mathbf{w}\| = \sqrt{w_1^2 + w_2^2}$$

This is simply Pythagoras' theorem applied to the weight vector. If you think of $w_1$ and $w_2$ as the horizontal and vertical sides of a right-angled triangle, $\|\mathbf{w}\|$ is the length of the hypotenuse, the straight-line distance from the origin to the tip of the arrow. Why does the length matter then? It turns out that $\|\mathbf{w}\|$ plays a central role in determining how good a decision boundary actually is, not just whether it separates the classes, but how confidently it does so.

The figure below makes the Pythagoras connection concrete. Drag the $w_1$ and $w_2$ sliders and watch the right-angled triangle update in real time: the blue horizontal side is $w_1$, the green vertical side is $w_2$, and the hypotenuse is $\|\mathbf{w}\|$. The formula panel on the left shows every step of the calculation with your current values substituted in.

Notice that making both $w_1$ and $w_2$ larger, while keeping their ratio the same, stretches the triangle outward without changing its angle. The hypotenuse gets longer but points in the same direction. This is precisely the distinction between the *direction* of $\mathbf{w}$ (which controls the orientation of the boundary) and its *length* $\|\mathbf{w}\|$ (which, as we will see, controls the margin).

In [ ]:
# Listing 26.
%matplotlib widget
from visualisations.Figure_13b import show
show()

---

We now have all the pieces in place to understand what makes an SVM different from every other linear classifier.

So far we have established that a hyperplane, defined by a weight vector $\mathbf{w}$ and a bias $b$, divides the feature space into two regions and classifies any new point by computing its score $\mathbf{w}^\top \mathbf{x} + b$ and checking whether it is positive or negative. We have also established that $\|\mathbf{w}\|$ is the length of the weight vector, and that it plays a central role in determining how good a boundary actually is.

But we have not yet answered the most important question: given that infinitely many hyperplanes could correctly separate two classes, which one should we actually use? Logistic regression, for example, also finds a linear boundary, but it does so by minimising classification error, and the boundary it lands on depends heavily on the exact distribution of the training data. Two slightly different training sets could produce quite different boundaries.

The SVM has a different and more principled answer. Consider two boundaries that both correctly separate the classes: one that passes very close to several training points, and one that sits neatly in the gap between the two classes with plenty of space on either side. The second boundary has more **breathing room**: if a new observation arrives that is slightly different from anything in the training set, a boundary with a large gap around it is far more likely to classify it correctly than one that hugs the training points tightly.

This gap, the space between the decision boundary and the nearest training points on each side, is called the **margin**. The SVM does not just find a boundary that separates the classes; it finds the boundary that maximises this margin. That is the **maximum margin** principle, and it is what gives SVMs both their name and their strong theoretical guarantees.

The figure below makes this concrete. The left panel shows two valid boundaries: one hugging Class −1 (the red dashed line), one hugging Class +1 (the blue dashed line). Both correctly separate all the training data. But look at what happens to the two star points: new observations that arrive after training, one from each class. If we use the red line as our decision boundary, the red star lands on the wrong side. If we use the blue line as our boundary, the blue star lands on the wrong side. Neither boundary has enough breathing room to handle data that sits in the gap between the classes.

The right panel shows the SVM's answer to this problem. The black boundary sits exactly halfway between the two classes, protected on either side by the grey shaded margin. The same two star points now fall comfortably on the correct side of the maximum margin boundary, because the central placement leaves enough space to absorb natural variation in new examples.

Notice also the grey dashed lines in the right panel. These are the margin boundaries, and the training points that sit exactly on them are the points that matter most. These boundary-touching points are called **support vectors**, and they are what the SVM is named after. Every other training point in the figure, every circle sitting well away from the boundary, plays no role in determining where the boundary sits. Remove them, and the solution does not change. It is the support vectors that help us compute the maximum margin boundary.


In [ ]:
# Listing 27.
%matplotlib widget
from visualisations.Figure_13c import show
show()

---

#### Why Maximise the Margin?

This is the step that trips most people up, and it deserves a careful explanation, because the relationship between $\|\mathbf{w}\|$ and the margin is counterintuitive at first glance.

Recall that $\|\mathbf{w}\|$ is the length of the weight vector. A larger $\|\mathbf{w}\|$ means a longer arrow. So it feels natural to think that a bigger, stronger weight vector would push the classes further apart and widen the gap. But this intuition is wrong, and understanding why is the key to understanding SVMs.

The issue is that $\mathbf{w}$ does two things simultaneously: it controls the *orientation* of the boundary (which direction it faces) and the *scale* of the score function (how quickly scores grow as you move away from the boundary). These two roles are coupled, and it is the second one that matters here.

Imagine you have a boundary that correctly separates the classes with some margin. Now scale up every value in $\mathbf{w}$ by a factor of 2: double $w_1$, double $w_2$, and adjust $b$ accordingly. The boundary does not move. The orientation does not change. But every score, the number $\mathbf{w}^\top \mathbf{x} + b$ computed for every data point, has also doubled. The data points haven't moved, but the numbers assigned to them have all grown larger.

Now here is the crucial point: the *physical distance* from a point to the boundary in feature space is not the same as its score. The score depends on both the distance and the scale of $\mathbf{w}$. If you double $\|\mathbf{w}\|$, the scores all double, but the actual distances between points and the boundary stay exactly the same. So a large $\|\mathbf{w}\|$ does not mean a wide margin. It just means the score function is measuring distances on a different scale.

The margin is a physical distance in feature space. It can be shown mathematically that this physical distance from the boundary to the nearest training points equals $\frac{1}{\|\mathbf{w}\|}$ on each side, giving a total margin of $\frac{2}{\|\mathbf{w}\|}$. This is an inverse relationship: as $\|\mathbf{w}\|$ grows, the margin shrinks; as $\|\mathbf{w}\|$ shrinks, the margin widens.

So to maximise the margin we want $\|\mathbf{w}\|$ to be as small as possible, subject to the constraint that the boundary still correctly separates all the training data. In practice the SVM minimises $\frac{1}{2}\|\mathbf{w}\|^2$ rather than $\|\mathbf{w}\|$ directly, since squaring removes the square root and makes the mathematics cleaner, but the solution is identical. The factor of $\frac{1}{2}$ is purely for notational convenience.

The entire learning process of a linear SVM can therefore be summarised simply: **find the $\mathbf{w}$ and $b$ that correctly separate the classes while keeping $\|\mathbf{w}\|$ as small as possible.** A smaller $\|\mathbf{w}\|$ means a wider physical gap between the classes, and a wider gap means better tolerance for new, unseen data.

---

The interactive figure below demonstrates this inverse relationship directly. The decision boundary, the black line, is fixed in place throughout. Dragging the $\|\mathbf{w}\|$ slider does not move the boundary or change its orientation. What it does change is the scale of the weight vector, and through it, the width of the margin.

The grey shaded region between the two dashed lines is the margin: the protected gap either side of the boundary. The equation panel shows the current $\|\mathbf{w}\|$ value and the resulting margin width $\frac{2}{\|\mathbf{w}\|}$, updating live.

**Drag the slider to the right** to increase $\|\mathbf{w}\|$, and watch the margin shrink. The dashed margin lines close in toward the boundary. The gap narrows.

**Drag the slider to the left** to decrease $\|\mathbf{w}\|$, and watch the margin widen. The dashed lines move outward. The gap grows.

Notice that the boundary itself never moves, only the margin changes. This is the key point: $\|\mathbf{w}\|$ does not control where the boundary sits. It controls how much breathing room surrounds it. A smaller $\|\mathbf{w}\|$ means a wider margin, which means the classifier is more tolerant of new data that does not land exactly where the training examples did. The SVM finds the smallest possible $\|\mathbf{w}\|$, the widest possible margin, subject to all training points being correctly classified.


In [ ]:
# Listing 28.
%matplotlib widget
from visualisations.Figure_13d import show
show()


---

#### The Hinge Loss: Handling Imperfect Data

So far we have assumed the ideal case: the two classes are perfectly separable by a straight line, and the SVM can find a hyperplane that correctly classifies every training point while maintaining a clear margin on both sides. In this ideal world, minimising $\|\mathbf{w}\|$ subject to correct classification is all we need.

But real data is rarely this clean. Classes overlap, labels are sometimes noisy, and in many practical problems no single straight line can correctly classify every training example. If we insist on perfect classification, if we demand that every point sits outside the margin on the correct side, the optimisation problem may have no solution at all. The SVM would fail before it even started.

The solution is to relax the requirement of perfect classification and instead allow some points to sit inside the margin, or even on the wrong side of the boundary, while penalising them for doing so. The penalty used is called the **hinge loss**.

For each training point $i$, the hinge loss measures how badly that point violates the margin:

$$\ell_i = \max(0,\ 1 - y_i(\mathbf{w}^\top \mathbf{x}_i + b))$$

Reading this term by term:

- $y_i$ is the true label of point $i$, either $+1$ or $-1$
- $\mathbf{w}^\top \mathbf{x}_i + b$ is the model's score for that point. A positive number means it is on the +1 side of the boundary, negative means it is on the −1 side
- $y_i(\mathbf{w}^\top \mathbf{x}_i + b)$ multiplies the true label by the score. When both have the same sign (the point is correctly classified), this product is positive. When they disagree (the point is on the wrong side), the product is negative.
- $1 - y_i(\mathbf{w}^\top \mathbf{x}_i + b)$ measures the degree of violation. A correctly classified point sitting well outside the margin gives a large positive product, making this term negative, meaning no violation. A point sitting inside the margin or on the wrong side gives a small or negative product, making this term positive, a violation.
- $\max(0, \ldots)$ clips the result so it can never be negative. A correctly classified point well outside the margin contributes zero loss; it is doing its job perfectly and needs no penalty. Only points that are too close to the boundary or on the wrong side incur a cost, and that cost grows linearly the further they violate the margin.

The hinge loss therefore behaves exactly as we would want: it ignores well-classified points and focuses the model's attention entirely on the difficult cases near or across the boundary.

---

The interactive figure below lets you see the hinge loss in action. The left panel shows the same decision boundary and margin from Figure 13c, with a query point, the star, that you can slide left and right along the horizontal dotted line. The right panel shows the full hinge loss curve across all possible positions of that point, with a marker tracking its current loss. The query point is a Class +1 example. Drag the slider and watch both panels update.

**Drag right** and the point moves into the blue region, well clear of the margin. Once it crosses the upper dashed margin line, the loss drops to exactly zero. The SVM considers this point safely classified; it is far enough from the boundary that it requires no penalty at all.

**Drag left toward the boundary** and as soon as the point crosses back inside the margin (the grey shaded region), the loss becomes positive even though the point is still on the correct side of the decision boundary. This is the key behaviour: the SVM does not just ask "is this point on the right side?" It asks "is this point far enough on the right side?" A point inside the margin is a partial violation, and it is penalised proportionally to how far inside the margin it sits.

**Drag further left across the boundary** and the point is now on the wrong side entirely. The loss continues to grow linearly. There is no cliff edge or sudden jump; the penalty simply keeps increasing the further the point strays.

Notice the three distinct regions on the loss curve in the right panel: a flat region at zero (safely outside the margin), a rising slope that begins at the margin boundary (partial violation), and a continued rise beyond the decision boundary (full misclassification). The transition from zero loss to positive loss happens at the margin boundary, not the decision boundary, and this is precisely what makes the SVM a margin classifier rather than just a boundary classifier.

The equation panel on the left shows the full calculation at every position, so you can see exactly how the score, the product $y_i \times \text{score}$, and the final loss are computed step by step.


In [ ]:
# Listing 29.
%matplotlib widget
from visualisations.Figure_13e import show
show()


---

#### The Full Training Objective

We now have two things the SVM needs to balance simultaneously. On one hand, we want to maximise the margin: keep $\|\mathbf{w}\|$ small so the gap between the classes is as wide as possible. On the other hand, we want to minimise classification errors: keep the hinge loss small so that as few points as possible sit inside the margin or on the wrong side of the boundary. In a perfect world these two goals would never conflict. In practice, on real messy data, they almost always do.

The SVM resolves this by combining both goals into a single expression, the full training objective, and introducing one tuning parameter, $C$, to control how they are balanced:

$$\text{Loss} = \underbrace{\frac{1}{2}\|\mathbf{w}\|^2}_{\text{maximise margin}} + C \sum_{i=1}^{n} \underbrace{\max(0,\, 1 - y_i(\mathbf{w}^\top \mathbf{x}_i + b))}_{\text{hinge loss per point}}$$

Reading this left to right: the first term keeps $\|\mathbf{w}\|$ small (which widens the margin). The second term adds up the hinge loss across every training point, the total penalty paid for all margin violations. $C$ is a single number that sits in front of the second term and controls how much weight the model gives to those violations relative to the margin.

Think of $C$ as a dial:

- **Turn $C$ down (small value)**: the violation penalty counts for less. The model is free to let some points sit inside the margin or even on the wrong side if doing so allows the margin to be wider. This produces a more relaxed, forgiving boundary that is less likely to overfit to individual noisy examples, sometimes called a *soft-margin* SVM.

- **Turn $C$ up (large value)**: the violation penalty counts for more. The model becomes increasingly reluctant to tolerate any margin violations, pushing the boundary to correctly classify every training point even if that means narrowing the margin substantially. This produces a tighter, harder boundary that is more sensitive to individual examples and more likely to overfit.

Neither extreme is universally correct. A very small $C$ may produce a boundary that is too tolerant, ignoring genuine structure in the data. A very large $C$ may produce a boundary that has essentially memorised the training examples. Choosing $C$ well, typically by trying several values and comparing performance on a held-out validation set, is the primary practical tuning task when fitting an SVM.

$C$ is the lever that controls the tradeoff between fitting the training data closely and generalising to new data. The specific form changes from model to model, but the underlying tension is always the same.

One point worth noting: unlike logistic regression, which finds its parameters through iterative gradient descent, and unlike Naïve Bayes, which simply counts and divides, SVMs are trained by solving a **convex quadratic optimisation problem**. The mathematical details are beyond the scope of this notebook, but the important practical consequence is that this class of problem is extremely well understood. It is guaranteed to have a unique solution, and that solution is always the global optimum. There are no local minima to get stuck in, no sensitivity to starting values, and no need to run the optimiser multiple times. Whatever the data, the SVM will always find the same best answer.

---


The interactive figure below shows a linear SVM fitted to a two-dimensional dataset with two classes, blue points (Class +1) and red points (Class −1). The dataset is deliberately well-separated so the core concepts are visible without distraction.

**What you are looking at:**

- **Solid black line**: the decision boundary, the hyperplane $\mathbf{w}^\top \mathbf{x} + b = 0$ that the SVM has learned. Points on one side are classified as +1; points on the other side as −1.
- **Dashed grey lines**: the margin boundaries at $\mathbf{w}^\top \mathbf{x} + b = \pm 1$. The SVM actively maximises the gap between these two lines.
- **Shaded grey region**: the margin itself. The wider this region, the more confident and generalisable the classifier is expected to be.
- **Gold rings**: the support vectors, the training points that sit exactly on the margin boundaries. These are the only points that determine where the boundary sits; every other point could be removed and the solution would not change.
- **Equation panel**: shows the current values of $\mathbf{w}$, $b$, $\|\mathbf{w}\|$, the margin width $\frac{2}{\|\mathbf{w}\|}$, the hinge loss, and the total loss. All update live as you move the slider.


**The C slider: controlling the margin width**

Drag the $C$ slider and watch what happens to the boundary and the equation panel:

- **Small C (drag left)**: the SVM prioritises a wide margin over correct classification. The boundary may allow some points to sit inside the margin or even on the wrong side; these are called **margin violations**. The hinge loss term in the equation panel will be non-zero, but the margin width will be large. This is a *soft-margin* SVM.
- **Large C (drag right)**: the SVM insists on correctly classifying every point and pushes the margin as narrow as needed to achieve this. The hinge loss drops toward zero but $\|\mathbf{w}\|$ grows and the margin shrinks. This increases the risk of overfitting.

Notice that the number of support vectors also changes with $C$: a softer margin tends to pull more points onto the margin boundary, while a hard margin concentrates the solution on fewer points.


**Adding noisy points: what happens at the boundary?**

Click **Add noisy point** to introduce points that sit on the wrong side of the margin, red points placed in Class +1 territory. These represent real-world scenarios where observations are mislabelled, ambiguous, or genuinely lie in the overlapping region between two classes.

With a **small C**, watch how the boundary barely reacts: the SVM accepts the violation and maintains its wide margin. The hinge loss increases but the boundary remains stable. This is the soft-margin SVM tolerating noise gracefully.

With a **large C**, the boundary shifts noticeably to try to correctly classify the noisy point, sometimes dramatically so. The SVM is being pulled by a single outlier, which is a classic symptom of overfitting to noise.

Keep adding points and adjusting $C$ to find the value where the boundary remains stable and sensible despite the noisy examples; this is the practical task of SVM hyperparameter tuning.


In [ ]:
# Listing 30.
%matplotlib widget
from visualisations.Figure_13f import show
show()

### 8.2 SVMs with scikit-learn

Scikit-learn's `SVC` (Support Vector Classifier) implements the full SVM training objective described above: finding the maximum margin hyperplane while balancing margin violations through the parameter $C$. The interface follows the same fit/predict pattern used throughout this series of notebooks, with one important addition: SVMs require features to be scaled before fitting.

Scaling transforms each feature so that it occupies a comparable numerical range. Without it, a feature measured in thousands (e.g. altitude in metres) would dominate one measured in single digits (e.g. a humidity index from 0-5), skewing the model's decisions purely due to units rather than genuine predictive power. A common approach is **standardisation**, which rescales each feature to have a mean of 0 and a standard deviation of 1, so a value that was 2,000 in the original data might become 0.3 after scaling, sitting on the same footing as everything else.

The dataset we will use contains 5,000 credit application records, each described by six financial features and a binary risk label. The table below summarises the features and their meanings.

| Feature | Description |
|---|---|
| `income` | Annual income (£k) |
| `debt_ratio` | Proportion of income committed to existing debt |
| `credit_score` | Credit score (300-850) |
| `loan_amount` | Loan amount requested (£k) |
| `employment_years` | Years in current employment |
| `missed_payments` | Number of missed payments in the past 12 months |
| `risk` | Class label: `0` = Low risk, `1` = High risk |

Each row represents a single applicant. The goal is to learn a decision boundary that separates low-risk applicants from high-risk ones using the six input features. Notice that the features span very different numerical ranges, `credit_score` runs from 300-850 while `debt_ratio` sits between 0 and 1, which is precisely why scaling is essential before fitting an SVM.

---

#### Training the Model

The code sample below shows you how to build a complete SVM pipeline in four steps. We load the `credit.csv` dataset and separate it into a feature matrix `X` (the six input measurements) and a label vector `y` (the risk category for each applicant). We then split these into a training set (75%) and a held-out test set (25%), using `stratify=y` to ensure both splits contain a proportional mix of each class.

Next we scale the features using `StandardScaler`. Notice that we call `fit_transform` on the training data but only `transform` on the test data; this is important. Fitting the scaler on the full dataset would leak information about the test set into training, giving an overly optimistic picture of performance.

Finally, we create an `SVC` with a linear kernel and fit it to the scaled training data. The `C` parameter controls the trade-off between finding the widest possible margin and tolerating misclassified training points; we'll explore its effect shortly. Setting `probability=True` allows us to retrieve confidence scores alongside predictions, at a small computational cost.

```python
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. Load the dataset
df = pd.read_csv('data/credit.csv')

FEAT_NAMES  = ['income', 'debt_ratio', 'credit_score',
               'loan_amount', 'employment_years', 'missed_payments']
CLASS_NAMES = ['Low risk', 'High risk']

X = df[FEAT_NAMES].values
y = df['risk'].values

# 2. Split into training and test sets
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

# 3. Scale the features
#    SVMs are sensitive to feature scale. Without this, features measured
#    on large numeric ranges (e.g. credit_score: 300–850) will dominate
#    the margin calculation regardless of their actual predictive value.
#    IMPORTANT: fit the scaler on training data only, then apply to both sets.
#    Fitting on the full dataset leaks test information into training.
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)   # fit AND transform training data
X_te_sc = scaler.transform(X_te)       # transform using training statistics only

# 4. Create and fit the SVM
#    kernel='linear' finds the maximum margin hyperplane as described in 8.1
#    C=1.0 is a reasonable default starting point
#    probability=True enables predict_proba(), adds a small computational cost
svm = SVC(kernel='linear', C=1.0, probability=True, random_state=0)
svm.fit(X_tr_sc, y_tr)
```

---

#### What the Model Has Learned

After calling `fit()`, the trained SVM exposes several useful attributes:

```python
svm.coef_            # weight vector w (one value per feature)
svm.intercept_       # bias term b
svm.support_vectors_ # the actual training examples identified as support vectors
svm.n_support_       # number of support vectors per class
```

- `coef_` contains the weight vector $\mathbf{w}$. A large positive weight means that feature pushes a point toward the high-risk side; a large negative weight pushes toward low risk. A weight near zero means the SVM found that feature largely unhelpful. Because the features are standardised, the weights are directly comparable across features; you do not need to worry about one feature dominating simply because it was measured on a larger scale.
- `n_support_` tells you how many training examples ended up as support vectors per class. A large number of support vectors relative to the training set size is a sign that the margin is soft and the classes overlap substantially.

| Attribute | What it contains |
|---|---|
| `svm.coef_` | The weight vector $\mathbf{w}$, one value per feature |
| `svm.intercept_` | The bias term $b$ |
| `svm.support_vectors_` | The actual training examples identified as support vectors |
| `svm.n_support_` | Number of support vectors per class |

---

#### Making Predictions and Getting Probabilities

```python
# Predict the class label for each test applicant
#   Each applicant is classified as low risk (0) or high risk (1)
y_pred = svm.predict(X_te_sc)

# Predict the probability of each class for each test applicant
#   Returns a matrix of shape (n_samples, 2)
#   Each row sums to 1
y_proba = svm.predict_proba(X_te_sc)
```

`predict_proba()` returns the estimated probability of each class. A row of `[0.12, 0.88]` means the model assigns 88% probability to High Risk. For a credit officer, this is far more actionable than a bare label: a borderline case at 52% warrants a very different response than one at 95%.

---

#### Evaluating the Model

Here we evaluate the trained SVM on the held-out test set. We report overall accuracy, then scikit-learn's `classification_report`, which breaks performance down by class, giving precision, recall, and F1-score for both low-risk and high-risk applicants separately. These per-class metrics are more informative than accuracy alone, particularly if the classes are imbalanced.

We also inspect the SVM's feature weights (`svm.coef_[0]`), which are available because we used a linear kernel. Each weight reflects how strongly, and in which direction, that feature pushes the model toward one class. A simple bar chart built from block characters makes the relative magnitudes easy to compare at a glance.

```python
from sklearn.metrics import accuracy_score, classification_report

print('SVM - Credit Risk Classification')
print('=' * 45)
print(f'Training samples : {len(X_tr_sc)}')
print(f'Test samples     : {len(X_te_sc)}')
print(f'Test accuracy    : {accuracy_score(y_te, y_pred):.1%}')
print()
print(classification_report(y_te, y_pred, target_names=CLASS_NAMES))

# Feature weights: shows which features the SVM weighted most heavily
print('Feature weights (w):')
for name, weight in zip(FEAT_NAMES, svm.coef_[0]):
    bar  = '█' * int(abs(weight) * 15)
    sign = '+' if weight >= 0 else '-'
    print(f'  {name:<20}  {sign}{abs(weight):.4f}  {bar}')
```

---

#### Key Hyperparameters

Here are the key parameters for the scikit-learn implementation of the SVM.

| Parameter | Default | What it controls |
|---|---|---|
| `kernel` | `'rbf'` | The kernel function: `'linear'` for a straight boundary, `'rbf'` for a curved one (covered in Section 8.3) |
| `C` | `1.0` | The margin-violation tradeoff: smaller values widen the margin and tolerate more errors; larger values narrow the margin and insist on correct classification |
| `gamma` | `'scale'` | Controls the reach of the RBF kernel, relevant only for non-linear SVMs |
| `probability` | `False` | Set to `True` to enable `predict_proba()`, adds a small computational cost |
| `class_weight` | `None` | Set to `'balanced'` to compensate for imbalanced classes |
| `random_state` | `None` | Set for reproducibility when `probability=True` |

The most important parameter to tune is `C`. A simple manual sweep across a range of values, comparing performance on the held-out test set, is a reasonable starting point:

```python
for c in [0.01, 0.1, 1.0, 10.0, 100.0]:
    m = SVC(kernel='linear', C=c, random_state=0)
    m.fit(X_tr_sc, y_tr)
    acc = accuracy_score(y_te, m.predict(X_te_sc))
    print(f'C={c:<8}  →  accuracy={acc:.4f}')
```

---

Bringing this all together, here's how to use scikit-learn to build and test a model on the credit data:

In [ ]:
# Listing 31.
# ── SVM — Credit Risk Classification ─────────────────────────────────────────
#
# Loads the synthetic credit risk dataset from data/credit.csv and trains
# a linear SVM to classify loan applicants as low risk (0) or high risk (1).
# Features are standardised before fitting — essential for SVMs.

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('data/credit.csv')

FEAT_NAMES  = ['income', 'debt_ratio', 'credit_score',
               'loan_amount', 'employment_years', 'missed_payments']
CLASS_NAMES = ['Low risk', 'High risk']

X = df[FEAT_NAMES].values
y = df['risk'].values

# ── Train / test split ────────────────────────────────────────────────────────
# stratify=y ensures both classes are proportionally represented in both sets.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

# ── Scale features ────────────────────────────────────────────────────────────
# SVMs measure the margin geometrically in feature space — without scaling,
# features on large numeric ranges (e.g. credit_score: 300–850) will dominate
# the distance calculations regardless of their actual predictive value.
# The scaler is fitted on training data only, then applied to both sets.
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)   # fit AND transform training data
X_te_sc = scaler.transform(X_te)       # apply training statistics to test data

# ── Fit the SVM ───────────────────────────────────────────────────────────────
# kernel='linear' — finds the maximum margin hyperplane as described in 8.1
# C=1.0           — default tradeoff between margin width and violations
# probability=True — enables predict_proba() for confidence scores
svm = SVC(kernel='linear', C=1.0, probability=True, random_state=0)
svm.fit(X_tr_sc, y_tr)

# ── Inspect what the model has learned ───────────────────────────────────────
print('SVM - Credit Risk Classification')
print('=' * 50)
print(f'Training samples         : {len(X_tr_sc)}')
print(f'Test samples             : {len(X_te_sc)}')
print(f'Support vectors — low risk  : {svm.n_support_[0]}')
print(f'Support vectors — high risk : {svm.n_support_[1]}')
print(f'Total support vectors    : {sum(svm.n_support_)} '
      f'({sum(svm.n_support_)/len(X_tr_sc):.1%} of training set)')
print()

# Feature weights — comparable across features because data is standardised.
# Positive weight → pushes toward high risk (class 1)
# Negative weight → pushes toward low risk (class 0)
print('Feature weights (w):')
print('-' * 45)
order = np.argsort(np.abs(svm.coef_[0]))[::-1]
for i in order:
    name   = FEAT_NAMES[i]
    weight = svm.coef_[0][i]
    bar    = '█' * int(abs(weight) * 20)
    sign   = '+' if weight >= 0 else '-'
    print(f'  {name:<20}  {sign}{abs(weight):.4f}  {bar}')

# ── Predict and evaluate ──────────────────────────────────────────────────────
y_pred  = svm.predict(X_te_sc)
y_proba = svm.predict_proba(X_te_sc)

print()
print(f'Test accuracy : {accuracy_score(y_te, y_pred):.1%}')
print()
print(classification_report(y_te, y_pred, target_names=CLASS_NAMES))

# ── Show a sample of predictions with confidence scores ───────────────────────
# For a credit officer, the probability is more useful than the bare label.
print('Sample predictions (first 8 test applicants):')
print('-' * 55)
print(f'  {"True":>10}  {"Predicted":>10}  {"P(Low risk)":>12}  {"P(High risk)":>12}')
print(f'  {"-"*10}  {"-"*10}  {"-"*12}  {"-"*12}')
for i in range(8):
    true_lbl = CLASS_NAMES[y_te[i]]
    pred_lbl = CLASS_NAMES[y_pred[i]]
    p_low    = y_proba[i][0]
    p_high   = y_proba[i][1]
    flag     = '  ← misclassified' if y_te[i] != y_pred[i] else ''
    print(f'  {true_lbl:>10}  {pred_lbl:>10}  {p_low:>11.1%}  {p_high:>11.1%}{flag}')

# ── C sweep — find the best value on the test set ────────────────────────────
print()
print('C sweep — effect on test accuracy:')
print('-' * 35)
for c in [0.01, 0.1, 1.0, 10.0, 100.0]:
    m = SVC(kernel='linear', C=c, random_state=0)
    m.fit(X_tr_sc, y_tr)
    acc = accuracy_score(y_te, m.predict(X_te_sc))
    bar = '█' * int(acc * 40)
    print(f'  C={c:<8}  accuracy={acc:.1%}  {bar}')

---

Here is a plot of the confusion matrix produced by this classifier.

In [ ]:
# Listing 32.
# ── Confusion matrix — SVM Credit Risk Classification ─────────────────────────
# Visualise model performance by displaying actual vs predicted class distribution
# with both raw counts and row-normalized percentages for better interpretability.

# Create figure with customizable canvas settings
fig, ax = plt.subplots(figsize=(6, 5))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

# Compute confusion matrix from true (y_te) and predicted (y_pred) labels
cm = confusion_matrix(y_te, y_pred)

# Create display object with class names for labeling
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)

# Plot the confusion matrix without colorbar, using a blue color gradient
disp.plot(ax=ax, colorbar=False, cmap='Blues')

# Calculate row-normalized percentages (each row sums to 100%)
# This shows what percentage of each true class was predicted as each class
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

# Annotate each cell with the corresponding percentage
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        # Position text slightly below the cell center (i + 0.28)
        ax.text(j, i + 0.28, f'({cm_norm[i, j]:.1%})',
                ha='center', va='center', fontsize=9,
                # Use white text for dark backgrounds, dark text for light backgrounds
                color='white' if cm_norm[i, j] > 0.6 else '#333')

# Add descriptive title and axis labels
ax.set_title('Confusion matrix — SVM Credit Risk Classification\n'
             'Rows = true class   |   Columns = predicted class',
             fontsize=10)
ax.set_xlabel('Predicted class', fontsize=11)
ax.set_ylabel('True class', fontsize=11)

plt.tight_layout()
plt.show()

---

### 8.3 Kernels - Non-Linear Boundaries in Higher Dimensions

A linear SVM works by finding the best possible straight-line boundary between classes. In two dimensions this boundary is a line, in three dimensions it is a plane, and in higher dimensions a **hyperplane**. This approach is highly effective when the classes can genuinely be separated by a linear boundary. Unfortunately, many real datasets do not have such a simple structure.

Consider a dataset consisting of two concentric rings: one class forms a small circle in the centre, while the other forms a larger circle surrounding it. No matter how a straight line is drawn, it is impossible to separate the two classes correctly. Similar problems arise in applications involving curved boundaries, nested clusters, and complex patterns that cannot be captured by a single linear decision boundary. The difficulty is not necessarily that the classes are impossible to separate, but that they cannot be separated **in their current representation**. 

One possible solution is to create new features that describe the data differently. If the observations could somehow be represented in a higher-dimensional space, patterns that appear tangled in two dimensions may become much easier to separate. Returning to the concentric-ring example, points that belong to different rings could be mapped to different regions of a higher-dimensional space, where a simple flat boundary might successfully divide the classes. The challenge is that explicitly transforming every observation into a high-dimensional feature space can be computationally expensive, particularly for large datasets. **Kernel functions** provide an elegant solution. Rather than performing the transformation directly, a kernel computes the similarity between observations in a way that has the same effect as working in the higher-dimensional space. This allows an SVM to learn complex, non-linear decision boundaries without the cost of explicitly constructing the transformed features.

This is where the **kernel trick** becomes powerful. To find an optimal separating hyperplane, an SVM does not actually need the full coordinates of every observation in the transformed feature space. Instead, during training it repeatedly asks a much simpler question:

> *How similar are these two observations?*

A kernel function answers this question directly. It computes a similarity score between pairs of observations in a way that is mathematically equivalent to first transforming the data into a higher-dimensional space and then comparing them there. The remarkable consequence is that the expensive transformation never has to be carried out explicitly. An analogy is to imagine being asked to compare the distances between cities on a globe. One approach would be to build a detailed three-dimensional model of the Earth and measure everything directly. A much more efficient approach is to use a formula that gives the correct distance without constructing the model at all. The kernel trick follows a similar idea: it produces the information the SVM needs without requiring the higher-dimensional representation to be built. This allows the SVM to behave as though it is operating in a much richer feature space than the one originally observed. Patterns that appear inseparable in the original data may become separable after the implicit transformation, enabling the model to construct curved and highly flexible decision boundaries. Importantly, the optimisation problem remains essentially the same as for a linear SVM, meaning that much of the mathematical elegance and computational efficiency of the original algorithm is preserved.


Mathematically, a kernel function is written as

$$k(\mathbf{x}_i,\mathbf{x}_j),$$

where the value returned reflects the similarity between observations $(\mathbf{x}_i)$ and $(\mathbf{x}_j)$. Different kernel functions define similarity in different ways, leading to different types of decision boundaries.

The most commonly used kernels are shown below.

| Kernel | Formula | Typical use |
|----------|----------|----------|
| **Linear** | $\mathbf{x}_i^\top \mathbf{x}_j$ | Data that is approximately linearly separable or extremely high-dimensional, such as text classification. |
| **RBF (Gaussian)** | $\exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$ | General-purpose choice for complex non-linear datasets. |
| **Polynomial** | $(\mathbf{x}_i^\top \mathbf{x}_j + r)^d$ | Problems where interactions between features create polynomial-shaped boundaries. |

Among these, the **RBF (Gaussian) kernel** is by far the most widely used. Rather than constructing a single global boundary, the RBF kernel allows each training observation to exert influence over its surrounding region. The resulting decision boundary emerges from the combined influence of all training points and can bend, curve, and adapt to remarkably complex patterns. This flexibility explains why the RBF kernel is often considered the default choice when the underlying structure of the data is unknown.

A crucial parameter of the RBF kernel is $\gamma$ (gamma), which determines how far the influence of each training observation extends. When $\gamma$ is small, each point influences a large neighbourhood, producing a smooth and relatively simple decision boundary that captures broad trends in the data. As $\gamma$ increases, the influence of each point becomes more localised, allowing the boundary to follow increasingly fine details of the training data. While this added flexibility can improve training performance, excessively large values of $\gamma$ may cause the model to fit noise rather than genuine structure, leading to overfitting.

Another important parameter is $C$, the margin-violation tradeoff introduced earlier. Together, $\gamma$  and $C$ determine the complexity of an RBF-based SVM. While $\gamma$  controls the shape and flexibility of the decision boundary, $C$ controls how strictly the model attempts to classify the training data correctly. Finding an appropriate balance between these parameters is often the key to building a model that generalises well to unseen observations.

---

The code example below applies linear, polynomial, and RBF kernels to a ring-shaped dataset in which linear separation is impossible. The resulting visualisations illustrate how kernel methods transform an apparently unsolvable problem into one that can be separated by a hyperplane in a higher-dimensional feature space. The second example explores the effect of varying \(C\), demonstrating how the balance between margin maximisation and classification accuracy influences the resulting decision boundary.


In [ ]:
# Listing 33.
# ── SVM kernels: linear, RBF, polynomial on ring-shaped data ────────────────
# Demonstrate how different SVM kernels handle non-linearly separable data using
# a synthetic ring-shaped dataset (two concentric circles).

# Initialize random number generator for reproducibility
rng_sv2 = np.random.default_rng(21)
n_ring = 120  # Number of samples per ring

# Generate inner ring: points sampled at small radii (0 to 1.2)
r_inner = rng_sv2.uniform(0, 1.2, n_ring)
ang_in = rng_sv2.uniform(0, 2 * np.pi, n_ring)

# Generate outer ring: points sampled at larger radii (2.0 to 3.2)
r_outer = rng_sv2.uniform(2.0, 3.2, n_ring)
ang_out = rng_sv2.uniform(0, 2 * np.pi, n_ring)

# Convert polar coordinates (r, θ) to Cartesian (x, y) for both rings
# and stack them into a single feature matrix
X_ring = np.vstack([
    np.c_[r_inner * np.cos(ang_in), r_inner * np.sin(ang_in)],
    np.c_[r_outer * np.cos(ang_out), r_outer * np.sin(ang_out)],
])
# Labels: 0 for inner ring, 1 for outer ring
y_ring = np.array([0] * n_ring + [1] * n_ring)

# Define three SVM classifiers with different kernels
kernels = [
    ('linear', SVC(kernel='linear', C=1.0)),      # Linear decision boundary
    ('RBF',    SVC(kernel='rbf', C=1.0, gamma='scale')),  # Radial basis function
    ('poly',   SVC(kernel='poly', C=1.0, degree=3)),     # Polynomial kernel
]

# Create a 3-panel figure for side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'

# Create a fine meshgrid for visualizing decision boundaries
h = 0.05  # Step size for meshgrid
x0_min, x0_max = X_ring[:, 0].min() - 0.5, X_ring[:, 0].max() + 0.5
x1_min, x1_max = X_ring[:, 1].min() - 0.5, X_ring[:, 1].max() + 0.5
xx_r, yy_r = np.meshgrid(np.arange(x0_min, x0_max, h),
                          np.arange(x1_min, x1_max, h))

# Train each SVM and plot its decision boundary
for ax, (kname, clf) in zip(axes, kernels):
    # Fit the classifier to the ring data
    clf.fit(X_ring, y_ring)
    # Compute training accuracy
    acc = clf.score(X_ring, y_ring)  # score() returns mean accuracy
    # Predict class labels for every point in the meshgrid
    Z_r = clf.predict(np.c_[xx_r.ravel(), yy_r.ravel()]).reshape(xx_r.shape)

    # Plot filled decision regions with class colors
    ax.contourf(xx_r, yy_r, Z_r, alpha=0.15, colors=['steelblue', 'tomato'])
    # Plot decision boundary contours
    ax.contour(xx_r, yy_r, Z_r, colors='black', linewidths=1.0)

    # Plot training points for each class
    for cls, col, lbl in [(0, 'steelblue', 'Inner'), (1, 'tomato', 'Outer')]:
        m = y_ring == cls
        ax.scatter(X_ring[m, 0], X_ring[m, 1], color=col, s=30,
                   edgecolors='k', lw=0.3, alpha=0.8, label=lbl, zorder=3)

    # Add title with kernel name and accuracy
    ax.set_title(f'Kernel: {kname}\nAccuracy = {acc:.1%}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.legend(fontsize=8)
    ax.set_aspect('equal')  # Ensure circles are not distorted

# Add main title summarizing the key insight
plt.suptitle('Figure 9: SVM with different kernels on ring-shaped data\n'
             'Linear kernel fails; RBF and polynomial adapt to the curved boundary',
             fontsize=10, y=0.97)
plt.tight_layout()
plt.show()

**When to use SVMs:**

- High-dimensional data (text, genomics) where the number of features is large relative to sample count.
- Clear margin of separation between classes.
- Non-linear boundaries needed: use the RBF or polynomial kernel.
- Small to medium datasets where training time is not a constraint.

**When to avoid them:**

- Very large datasets: training time scales roughly as $O(n^2)$ to $O(n^3)$ with the number of samples.
- Noisy data with heavily overlapping classes.
- When interpretability is required.
- Multi-class problems (SVMs natively solve binary classification; multi-class requires combining several binary classifiers).


---


## 9. All Classifiers Side by Side

🔙 [Back to Table of Contents](#Table-of-Contents)

The **two-moons** dataset is a standard benchmark: two interleaved crescents that are clearly non-linearly separable. It directly tests whether a classifier can learn curved boundaries. Running all classifiers on the same dataset lets us see directly how the choice of model affects both the shape of the decision boundary and the resulting accuracy.

The code cell below trains six classifiers you've encountered so far, on the same training data and plots their decision regions for comparison.

In [ ]:
# Listing 34.
# ── Side-by-side comparison of all classifiers ───────────────────────────
# Two-moon data: two interleaved crescent shapes — a classic non-linear benchmark.
# This dataset is specifically designed to test a model's ability to learn non-linear decision boundaries.

# Generate synthetic two-moon dataset with 300 samples and moderate noise
X_m, y_m = make_moons(n_samples=300, noise=0.25, random_state=0)

# Split into training (75%) and test (25%) sets with fixed random state for reproducibility
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_m, y_m, test_size=0.25, random_state=0)

# Define classifiers to compare: linear, probabilistic, tree-based, and kernel methods
classifiers = [
    ('Logistic Regression',  LogisticRegression()),           # Linear model
    ('Naïve Bayes',          GaussianNB()),                   # Probabilistic linear model
    ('Decision Tree (d=4)',  DecisionTreeClassifier(max_depth=4, random_state=0)),  # Non-linear, depth-limited
    ('Random Forest',        RandomForestClassifier(n_estimators=100, random_state=0)),  # Ensemble of trees
    ('SVM (linear)',         SVC(kernel='linear', C=1.0)),   # Linear kernel SVM
    ('SVM (RBF)',            SVC(kernel='rbf', C=1.0, gamma='scale')),  # Non-linear RBF kernel
]

# Build a mesh over the feature space to evaluate and colour decision regions
# This allows visualisation of where each classifier draws its decision boundaries
h = 0.04  # Step size for the meshgrid
xlo, xhi = X_m[:, 0].min() - 0.5, X_m[:, 0].max() + 0.5  # x-axis bounds with padding
ylo, yhi = X_m[:, 1].min() - 0.5, X_m[:, 1].max() + 0.5  # y-axis bounds with padding
xx_m, yy_m = np.meshgrid(np.arange(xlo, xhi, h), np.arange(ylo, yhi, h))  # Create meshgrid

# Create 2x3 subplot grid for the 6 classifiers
fig, axes = plt.subplots(2, 3, figsize=(10, 8))
fig.canvas.header_visible = False
fig.canvas.resizable = True
fig.canvas.toolbar_visible = True
fig.canvas.toolbar_position = 'right'


# Train each classifier, predict on meshgrid, and plot decision boundaries
for ax, (name, clf) in zip(axes.ravel(), classifiers):
    clf.fit(X_tr_m, y_tr_m)  # Train on training data
    acc = accuracy_score(y_te_m, clf.predict(X_te_m))  # Evaluate on test set
    Z_m = clf.predict(np.c_[xx_m.ravel(), yy_m.ravel()]).reshape(xx_m.shape)  # Predict on meshgrid

    # Plot filled decision regions with light colors
    ax.contourf(xx_m, yy_m, Z_m, alpha=0.18, colors=['steelblue', 'tomato'])
    # Plot decision boundary contours
    ax.contour(xx_m, yy_m, Z_m, colors='white', linewidths=0.8)
    # Plot the actual data points
    for cls, col in [(0, 'steelblue'), (1, 'tomato')]:
        m = y_m == cls
        ax.scatter(X_m[m, 0], X_m[m, 1], color=col, s=25,
                   edgecolors='k', lw=0.25, alpha=0.7, zorder=3)
    # Add classifier name and test accuracy to title
    ax.set_title(f'{name}\nTest accuracy = {acc:.1%}', fontsize=10)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

# Add overall figure title summarizing the key insight
plt.suptitle('Figure 11: All classifiers on two-moon data\n'
             '(linear models struggle; ensemble and kernel methods adapt)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Print formatted summary table of test accuracies for all classifiers
print('Summary of test accuracies:')
print(f'  {"Classifier":<28}  Test accuracy')
print('-' * 45)
for name, clf in classifiers:
    acc = accuracy_score(y_te_m, clf.predict(X_te_m))
    print(f'  {name:<28}  {acc:.1%}')

---

**What the comparison reveals:**

The two-moons dataset contains a strongly non-linear class boundary, making it a useful test of how different classifiers handle complex geometric structure. The results reflect the strengths and limitations of each model family.

Logistic Regression (81.3%) and the linear SVM (82.7%) both assume that the classes can be separated by a single linear boundary. Because the two moons are interleaved rather than linearly separable, neither model can adequately capture the underlying pattern. Naïve Bayes performs slightly better (84.0%), but its assumption that each class follows a simple Gaussian distribution is also poorly matched to the crescent-shaped structure of the data.

The Decision Tree (85.3%) can model non-linear relationships, but it does so through a series of axis-aligned splits. As a result, it approximates the curved boundary using a collection of rectangular regions, producing a decision surface that is often irregular and less accurate than more flexible alternatives.

The two best-performing models are the Random Forest (96.0%) and the RBF SVM (94.7%). Both are capable of learning highly non-linear decision boundaries. The Random Forest achieves this by averaging the predictions of many trees, creating a smoother and more robust boundary than any individual tree. The RBF SVM achieves a similar goal through the kernel trick, which allows it to represent complex curved boundaries while still solving an optimisation problem based on a linear separator in a transformed feature space.

Importantly, these results do **not** imply that Random Forests or RBF SVMs are universally superior classifiers. They simply performed best on **this particular dataset**, whose geometry strongly favours models capable of learning non-linear boundaries. On a different dataset, for example, one that is approximately linearly separable, contains very high-dimensional sparse features, or has a much smaller sample size, the ranking could be entirely different. In such situations, Logistic Regression or a linear SVM might perform just as well, or even better.

This illustrates a fundamental principle of machine learning: model performance depends on how well a model's assumptions align with the structure of the data. The RBF SVM was not successful because it is inherently a "better" algorithm; it was successful because its assumptions happened to match the characteristics of the two-moons dataset. There is no single algorithm that is optimal for every possible problem. This idea is formalised by the **No Free Lunch Theorem**, which states that no learning algorithm can outperform all others across all possible datasets. The goal of model selection is therefore not to find the universally "best" classifier, but to identify the classifier whose assumptions are most appropriate for the problem at hand.



---

## 10. Summary

🔙 [Back to Table of Contents](#Table-of-Contents)

The table below summarises the four classifiers introduced here alongside logistic regression (covered in Notebook 5) for comparison.

| Classifier | Family | Objective | Key hyperparameters | Typical strengths |
|------------|--------|-----------|---------------------|-------------------|
| **Logistic Regression** | Discriminative | Log-loss (cross-entropy) | `C` (regularisation) | Linear boundaries; fast; interpretable weights |
| **Naïve Bayes** | Generative | MLE of $P(\mathbf{X}, Y)$ | `var_smoothing` | Text/high-dimensional data; fast; small datasets |
| **Decision Tree** | Discriminative | Gini impurity / entropy at each split | `max_depth`, `min_samples_leaf` | Interpretable rules; mixed data types; non-linear |
| **Random Forest** | Discriminative (ensemble) | Majority vote of randomised trees | `n_estimators`, `max_depth` | General purpose; built-in feature importance; robust to overfitting |
| **SVM (linear)** | Discriminative | Hinge loss + margin | $C$ | High-dimensional data; clear margin; text classification |
| **SVM (RBF)** | Discriminative | Hinge loss + kernel margin | $C$, `gamma` | Non-linear boundaries; medium-sized datasets |

Key concepts introduced in this notebook:

- **Generative vs discriminative**: generative models learn $P(\mathbf{X}, Y)$; discriminative models learn $P(Y \mid \mathbf{X})$ directly.
- **Naïve independence assumption**: conditionally independent features given class label; enables tractable computation at the cost of a potentially incorrect assumption.
- **Laplace smoothing**: prevents zero-probability issues in Naïve Bayes for unseen feature values.
- **Gini impurity**: measures class impurity at a decision tree node; minimised at each split by CART.
- **Overfitting in decision trees**: unlimited depth memorises training data; `max_depth` is the primary control.
- **Bootstrap sampling and random feature selection**: two sources of randomness that make Random Forest trees diverse and their errors decorrelated.
- **Maximum margin hyperplane**: the SVM decision boundary that maximises the distance to the nearest training points (support vectors).
- **Kernel trick**: computes inner products in a high-dimensional feature space without explicitly computing the transformation; enables non-linear boundaries.
- **No free lunch**: no single classifier is best for all datasets; model choice depends on data geometry, size, and the application's interpretability requirements.


---


## 11. References

🔙 [Back to Table of Contents](#Table-of-Contents)

Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.

Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324

Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning*, 20(3), 273–297. https://doi.org/10.1007/BF00994018

Mitchell, T. M. (1997). *Machine Learning*. McGraw-Hill.

Murphy, K. P. (2022). *Probabilistic Machine Learning: An Introduction*. MIT Press. Available at: https://probml.github.io/pml-book/book1.html

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830. http://jmlr.org/papers/v12/pedregosa11a.html

Quinlan, J. R. (1986). Induction of decision trees. *Machine Learning*, 1(1), 81–106. https://doi.org/10.1007/BF00116251